In [ ]:
# In[1]:

import pandas as pd
import pytz

# Load CSVs (they are in the current working directory)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Ensure value columns are numeric where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Convert timestamps to timezone-aware datetimes in UTC+8 (Asia/Shanghai) for consistency
tz = pytz.timezone('Asia/Shanghai')
for df in (df_metric, df_trace, df_log, df_error):
    # Some files might not have timestamp column — but per schema they do.
    if 'timestamp' in df.columns:
        # timestamps are in seconds (UTC); convert to UTC then to Asia/Shanghai
        dt = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
        df['timestamp_dt'] = dt

# 1) Metric aggregations
g = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
agg_metric = g.agg(
    sample_count='count',
    p95=lambda x: x.quantile(0.95),
    mean='mean',
    min='min',
    max='max'
).reset_index()

agg_metric_sorted = agg_metric.sort_values('sample_count', ascending=False).reset_index(drop=True)

metric_unique_cmdb = sorted(df_metric['cmdb_id'].dropna().unique().tolist())
metric_total_rows = df_metric.shape[0]

# 2) Trace aggregations
g = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
agg_trace = g.agg(
    sample_count='count',
    p95=lambda x: x.quantile(0.95),
    mean='mean',
    min='min',
    max='max'
).reset_index()

agg_trace_sorted = agg_trace.sort_values('sample_count', ascending=False).reset_index(drop=True)

trace_unique_cmdb = sorted(df_trace['cmdb_id'].dropna().unique().tolist())
trace_total_rows = df_trace.shape[0]

# 3) Log aggregations
g = df_log.groupby(['cmdb_id', 'log_name'])['value']
agg_log = g.agg(
    sample_count='count',
    p95=lambda x: x.quantile(0.95),
    mean='mean',
    min='min',
    max='max'
).reset_index()

agg_log_sorted = agg_log.sort_values('sample_count', ascending=False).reset_index(drop=True)

log_unique_cmdb = sorted(df_log['cmdb_id'].dropna().unique().tolist())
log_total_rows = df_log.shape[0]

# 4) Error logs summary: counts per cmdb_id, earliest timestamp (UTC+8)
# Limit to columns (timestamp, cmdb_id) per instructions when grouping
df_error_small = df_error.loc[:, ['timestamp', 'cmdb_id']].copy()
# Add converted datetime if not present
df_error_small['timestamp_dt'] = pd.to_datetime(df_error_small['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

err_grp = df_error_small.groupby('cmdb_id').agg(
    error_count=('timestamp', 'size'),
    earliest_ts=('timestamp_dt', 'min')
).reset_index().sort_values('error_count', ascending=False).reset_index(drop=True)

error_total_rows = df_error.shape[0]

# Prepare compact outputs (each limited to top 20 rows)
metric_summary = agg_metric_sorted.head(20)
trace_summary = agg_trace_sorted.head(20)
log_summary = agg_log_sorted.head(20)
error_summary = err_grp.head(20)

# Display the compact summaries and the requested auxiliary info.
# Use variable names (IPython will display them). Each line is compact and limited to 20 rows.
metric_summary, metric_unique_cmdb, metric_total_rows, trace_summary, trace_unique_cmdb, trace_total_rows, log_summary, log_unique_cmdb, log_total_rows, error_total_rows, error_summary

```
Out[1]:
```
Summary of the loaded telemetry and aggregation results:

- metric.csv
  - Total rows: 27,812
  - Unique cmdb_id count: 30 (examples: IG01, IG02, MG01, MG02, Mysql01, Mysql02, Redis01, Redis02, Tomcat01–Tomcat04, apache01–apache02, dockerA1/A2, dockerB1/B2, ServiceTest1–11)
  - Top aggregated groups (by sample_count = 31): examples include
    - Mysql01 · OSLinux_Host_Uptime (sample_count 31)
    - Tomcat01 · many Tomcat-Threads / Tomcat-Sessions KPIs (sample_count 31)
    - ServiceTest7 / ServiceTest8 app.* metrics (sample_count 31)
  - For each top group we computed: sample_count, P95, mean, min, max. The full aggregated table is stored as variable agg_metric.

- trace.csv
  - Total rows: 6,870
  - Unique cmdb_id count: 12 (examples: dockerA1/A2, dockerB1/B2, IG01/IG02, MG01/MG02, Tomcat01–Tomcat04)
  - Top aggregated groups (by sample_count = 30): dominated by docker nodes, e.g.
    - dockerB2 · trace.to_MG02.row_count (sample_count 30)
    - dockerA2 · trace.self.row_count, trace.to_MG01.*, trace.from_MG02.* (sample_count 30)
  - For each top group we computed: sample_count, P95, mean, min, max. The full aggregated table is stored as variable agg_trace.

- log.csv
  - Total rows: 438
  - Unique cmdb_id count: 7 (Mysql01, Tomcat01–Tomcat04, apache01, apache02)
  - Top aggregated groups (by sample_count up to 31): examples
    - Tomcat01 · log.row_count (sample_count 31; P95 ≈145.5, mean ≈120.48)
    - apache02 · log.row_count (sample_count 31; P95 ≈289.5, mean ≈238.81)
    - apache01 · log.row_count (sample_count 31; P95 ≈287.0, mean ≈239.94)
    - log.error_count entries for several servers show zero mean and zero P95 in top groups
  - For each top group we computed: sample_count, P95, mean, min, max. The full aggregated table is stored as variable agg_log.

- error_logs.csv
  - Total rows: 0
  - No error log rows found, so the per-cmdb_id error counts are empty.

If you want, I can (a) show the stored aggregated DataFrame variables (agg_metric / agg_trace / agg_log) limited to selected columns, (b) provide the exact top-20 rows as tables again, or (c) compute additional summaries (e.g., highest P95s across metrics/traces/logs). Which would you prefer next?

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id                                           kpi_name  sample_count           p95          mean           min          max
0        Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime            31  6.585116e+06  6.584303e+06  6.583400e+06  6585207.000
1       Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...            31  5.000000e+02  5.000000e+02  5.000000e+02      500.000
2       Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
3       Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
4       Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
5       Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
6       Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
7       Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
8       Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...            31  0.000000e+00  0.000000e+00  0.000000e+00        0.000
9       Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...            31  1.500000e+00  4.193548e-01  0.000000e+00        2.000
10       Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount            31  7.000000e+00  7.000000e+00  7.000000e+00        7.000
11       Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...            31  4.000000e-01  3.967742e-01  3.000000e-01        0.400
12  ServiceTest7                                             app.rr            31  1.000000e+02  1.000000e+02  1.000000e+02      100.000
13  ServiceTest7                                             app.sr            31  1.000000e+02  1.000000e+02  1.000000e+02      100.000
14  ServiceTest4                                            app.cnt            31  7.450000e+01  4.170968e+01  1.200000e+01       80.000
15      Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...            31  3.300540e+05  3.300540e+05  3.300540e+05   330054.000
16  ServiceTest8                                             app.sr            31  1.000000e+02  1.000000e+02  1.000000e+02      100.000
17  ServiceTest8                                             app.rr            31  1.000000e+02  1.000000e+02  1.000000e+02      100.000
18  ServiceTest8                                            app.mrt            31  2.368700e-01  1.468477e-01  4.158000e-02        1.048
19  ServiceTest8                                            app.cnt            31  7.600000e+01  4.019355e+01  1.000000e+00       80.000, ['IG01', 'IG02', 'MG01', 'MG02', 'Mysql01', 'Mysql02', 'Redis01', 'Redis02', 'ServiceTest1', 'ServiceTest10', 'ServiceTest11', 'ServiceTest2', 'ServiceTest3', 'ServiceTest4', 'ServiceTest5', 'ServiceTest6', 'ServiceTest7', 'ServiceTest8', 'ServiceTest9', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 27812,      cmdb_id                     trace_name  sample_count         p95        mean         min         max
0   dockerB2        trace.to_MG02.row_count            30   90.050000   54.133333   16.000000  109.000000
1   dockerB2     trace.to_MG02.duration_p95            30    0.184530    0.029797    0.007000    0.196250
2   dockerB2    trace.to_MG02.duration_mean            30    0.044233    0.019061    0.005581    0.082825
3   dockerA2       trace.self.duration_mean            30    0.024952    0.016038    0.006042    0.031868
4   dockerA2        trace.self.duration_p95            30    0.012000    0.008957    0.002450    0.012500
5   dockerA2           trace.self.row_count            30  844.150000  621.166667  403.000000  872.000000
6   dockerA2    trace.to_MG01.duration_mean            30    0.093000    0.019521    0.005800    0.138875
7   dockerA2     trace.to_MG01.duration_p95            30    0.473307    0.080970    0.006000    1.032750
8   dockerA2        trace.to_MG01.row_count            30   21.100000   11.866667    3.000000   27.000000
9   dockerA2    trace.to_MG02.duration_mean            30    0.085943    0.020210    0.005818    0.125789
10  dockerA2     trace.to_MG02.duration_p95            30    0.420450    0.086972    0.008000    1.017000
11  dockerA2   trace.from_MG01.duration_p95            30    1.056555    0.712005    0.078100    1.067000
12  dockerA2      trace.from_MG01.row_count            30   61.650000   44.033333   29.000000   64.000000
13  dockerA2  trace.from_MG02.duration_mean            30    0.267383    0.151837    0.058733    0.333306
14  dockerA2   trace.from_MG02.duration_p95            30    1.119900    0.774982    0.078000    1.371000
15  dockerA2      trace.from_MG02.row_count            30   61.750000   43.600000   28.000000   67.000000
16  dockerA1           trace.self.row_count            30  811.400000  580.800000  365.000000  850.000000
17  dockerA1    trace.to_MG01.duration_mean            30    0.066047    0.013699    0.005500    0.079364
18  dockerA1     trace.to_MG01.duration_p95            30    0.378657    0.048633    0.006350    0.410000
19  dockerA1        trace.to_MG01.row_count            30   19.200000   10.800000    1.000000   26.000000, ['IG01', 'IG02', 'MG01', 'MG02', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2'], 6870,      cmdb_id          log_name  sample_count       p95        mean    min     max
0   Tomcat01     log.row_count            31  145.5000  120.483871   1.00  152.00
1   Tomcat01   log.error_count            31    0.0000    0.000000   0.00    0.00
2   apache02     log.row_count            31  289.5000  238.806452   4.00  299.00
3   apache01     log.row_count            31  287.0000  239.935484   2.00  313.00
4   Tomcat03   log.error_count            31    0.0000    0.000000   0.00    0.00
5   apache02   log.error_count            31    0.0000    0.000000   0.00    0.00
6   apache01   log.error_count            31    0.0000    0.000000   0.00    0.00
7   Tomcat04     log.row_count            31  154.5000  117.935484   3.00  166.00
8   Tomcat04   log.error_count            31    0.0000    0.000000   0.00    0.00
9   Tomcat03     log.row_count            31  150.5000  121.580645   2.00  152.00
10  Tomcat02   log.error_count            30    0.0000    0.000000   0.00    0.00
11  Tomcat02     log.row_count            30  152.0000  123.933333  83.00  154.00
12   Mysql01     log.row_count            16    4.0000    2.312500   1.00    4.00
13   Mysql01   log.error_count            16    0.0000    0.000000   0.00    0.00
14  Tomcat02  log.gc_pause_max            13    0.0800    0.065385   0.02    0.08
15  Tomcat01  log.gc_pause_max             8    0.0865    0.070000   0.04    0.09
16  Tomcat03  log.gc_pause_max             8    0.0800    0.060000   0.02    0.08
17  Tomcat04  log.gc_pause_max             7    0.0800    0.051429   0.02    0.08, ['Mysql01', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04', 'apache01', 'apache02'], 438, 0, Empty DataFrame
Columns: [cmdb_id, error_count, earliest_ts]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables: df_metric, agg_metric
# Ensure timezone-aware timestamp column exists for df_metric
tz = pytz.timezone('Asia/Shanghai')
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure tz-aware and in Asia/Shanghai
    if df_metric['timestamp_dt'].dt.tz is None:
        df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_metric['timestamp_dt'] = df_metric['timestamp_dt'].dt.tz_convert(tz)

# Define time window in UTC+8
start = pd.Timestamp('2021-03-10 22:00:00', tz=tz)
end = pd.Timestamp('2021-03-10 22:30:00', tz=tz)

# Filter df_metric to the time window (inclusive)
df_metric_window = df_metric[(df_metric['timestamp_dt'] >= start) & (df_metric['timestamp_dt'] <= end)].copy()

# Merge with precomputed agg_metric to get global p95 (agg_metric must exist)
# Ensure agg_metric has expected columns
if 'p95' not in agg_metric.columns:
    raise ValueError("agg_metric does not contain 'p95' column")

df_merge = df_metric_window.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomaly rows where value >= global p95 (respecting NaNs)
df_merge['is_anom'] = False
mask_valid = df_merge['p95'].notna()
df_merge.loc[mask_valid, 'is_anom'] = df_merge.loc[mask_valid, 'value'] >= df_merge.loc[mask_valid, 'p95']

# Aggregate: max_value (within window) and p95 (global)
group_all = df_merge.groupby(['cmdb_id', 'kpi_name']).agg(
    max_value=('value', 'max'),
    p95=('p95', 'first')
)

# Aggregate anomalies: count and earliest timestamp among anomaly rows
df_anom = df_merge[df_merge['is_anom']].copy()
if not df_anom.empty:
    group_anom = df_anom.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count=('is_anom', 'sum'),
        earliest_anom_ts=('timestamp_dt', 'min')
    )
else:
    # empty frame with correct columns
    group_anom = pd.DataFrame(columns=['anomaly_count', 'earliest_anom_ts'])

# Combine aggregates
metric_anomalies = group_all.join(group_anom, how='left').fillna({'anomaly_count': 0})
# Ensure proper dtypes
metric_anomalies['anomaly_count'] = metric_anomalies['anomaly_count'].astype(int)

# Compute max_over_p95_ratio, handle p95 == 0 or NaN
metric_anomalies['max_over_p95_ratio'] = np.nan
valid_p95 = metric_anomalies['p95'].notna() & (metric_anomalies['p95'] != 0)
metric_anomalies.loc[valid_p95, 'max_over_p95_ratio'] = metric_anomalies.loc[valid_p95, 'max_value'] / metric_anomalies.loc[valid_p95, 'p95']
# For p95 == 0 and max_value > 0, set ratio to inf
zero_p95_mask = metric_anomalies['p95'] == 0
metric_anomalies.loc[zero_p95_mask & (metric_anomalies['max_value'] > 0), 'max_over_p95_ratio'] = np.inf

# Keep only groups with anomaly_count > 0
metric_anomalies = metric_anomalies[metric_anomalies['anomaly_count'] > 0].reset_index()

# Format earliest_anom_ts as timezone-aware timestamps (already tz-aware). Keep as-is.
# Order columns as requested and sort by anomaly_count desc, limit top 20
metric_anomalies = metric_anomalies[['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anom_ts', 'max_value', 'p95', 'max_over_p95_ratio']]
metric_anomalies = metric_anomalies.sort_values('anomaly_count', ascending=False).reset_index(drop=True)

# Store metric_anomalies variable in kernel (already assigned). Display compact top 20 rows.
metric_anomalies.head(20)

```
Out[2]:
```
```python
# Summarize metric_anomalies succinctly in plain English as a single string output.
# Reuse metric_anomalies from the kernel state.

top_n = metric_anomalies.head(5)
total_groups = metric_anomalies.shape[0]

lines = []
lines.append(f"Total anomalous (cmdb_id, kpi_name) groups in the 2021-03-10 22:00–22:30 window: {total_groups}.")

if total_groups == 0:
    lines.append("No anomalies detected in the specified time window.")
else:
    lines.append("Top anomaly groups (by anomaly_count):")
    for i, row in top_n.iterrows():
        cmdb = row['cmdb_id']
        kpi = row['kpi_name']
        cnt = int(row['anomaly_count'])
        ts = row['earliest_anom_ts']
        maxv = row['max_value']
        p95 = row['p95']
        ratio = row['max_over_p95_ratio']
        # Represent NaN/inf nicely
        ratio_str = ("NaN" if pd.isna(ratio) else ("inf" if ratio == float('inf') else f"{ratio:.3f}"))
        lines.append(f"- {cmdb} | {kpi} | anomalies: {cnt} | earliest: {ts} | max: {maxv:.6g} | p95: {p95:.6g} | max/p95: {ratio_str}")
    # note about patterns seen
    # check how many groups have p95 == 0
    zero_p95_count = int((metric_anomalies['p95'] == 0).sum())
    lines.append(f"Note: {zero_p95_count} group(s) have global p95 == 0, producing undefined (NaN) ratios.")
    # check if many anomalies span full window (anomaly_count equals window length)
    max_count = int(metric_anomalies['anomaly_count'].max())
    lines.append(f"Max anomaly_count observed: {max_count}. Many top groups show anomaly_count = {max_count}, indicating the metric was at/above its global p95 for each minute in the window.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count          earliest_anom_ts     max_value           p95  max_over_p95_ratio
0       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed             31 2021-03-10 22:00:00+08:00  5.792518e+07  5.792518e+07                 1.0
1       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax             31 2021-03-10 22:00:00+08:00  2.040136e+09  2.040136e+09                 1.0
2   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
3   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
4   apache02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount             31 2021-03-10 22:00:00+08:00  7.000000e+00  7.000000e+00                 1.0
5    Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
6    Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
7    Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
8   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
9   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
10  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
11  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
12  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
13  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
14  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
15  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...             31 2021-03-10 22:00:00+08:00  3.050000e+02  3.050000e+02                 1.0
16  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...             31 2021-03-10 22:00:00+08:00  3.303510e+05  3.303510e+05                 1.0
17  Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
18  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...             31 2021-03-10 22:00:00+08:00  0.000000e+00  0.000000e+00                 NaN
19  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...             31 2021-03-10 22:00:00+08:00  1.000000e+02  1.000000e+02                 1.0```
```

In [ ]:
# In[3]:

import numpy as np
import pytz
tz = pytz.timezone('Asia/Shanghai')

# Ensure df_trace and agg_trace exist in kernel state
# Prepare timezone-aware timestamp column
if 'timestamp_dt' not in df_trace.columns:
    df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    # ensure converted to Asia/Shanghai
    if df_trace['timestamp_dt'].dt.tz is None:
        df_trace['timestamp_dt'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df_trace['timestamp_dt'] = df_trace['timestamp_dt'].dt.tz_convert(tz)

# Define time window (inclusive) in UTC+8
start = pd.Timestamp('2021-03-10 22:00:00', tz=tz)
end = pd.Timestamp('2021-03-10 22:30:00', tz=tz)

# Filter trace rows within the time window
df_trace_window = df_trace[(df_trace['timestamp_dt'] >= start) & (df_trace['timestamp_dt'] <= end)].copy()

# Merge with precomputed agg_trace to get global p95
if 'p95' not in agg_trace.columns:
    raise ValueError("agg_trace must contain 'p95' column")

df_t_merge = df_trace_window.merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Determine anomaly rows where value >= global p95 (only where p95 is valid)
df_t_merge['is_anom'] = False
valid_mask = df_t_merge['p95'].notna()
df_t_merge.loc[valid_mask, 'is_anom'] = df_t_merge.loc[valid_mask, 'value'] >= df_t_merge.loc[valid_mask, 'p95']

# Aggregate overall (max_value and p95)
group_all = df_t_merge.groupby(['cmdb_id', 'trace_name']).agg(
    max_value=('value', 'max'),
    p95=('p95', 'first')
)

# Aggregate anomalies: count and earliest anomaly timestamp
df_t_anom = df_t_merge[df_t_merge['is_anom']].copy()
if not df_t_anom.empty:
    group_anom = df_t_anom.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_count=('is_anom', 'sum'),
        earliest_anom_ts=('timestamp_dt', 'min')
    )
else:
    group_anom = pd.DataFrame(columns=['anomaly_count', 'earliest_anom_ts'])

# Combine and compute ratio
trace_anomalies = group_all.join(group_anom, how='left').fillna({'anomaly_count': 0})
trace_anomalies['anomaly_count'] = trace_anomalies['anomaly_count'].astype(int)

trace_anomalies['max_over_p95_ratio'] = np.nan
valid_p95 = trace_anomalies['p95'].notna() & (trace_anomalies['p95'] != 0)
trace_anomalies.loc[valid_p95, 'max_over_p95_ratio'] = trace_anomalies.loc[valid_p95, 'max_value'] / trace_anomalies.loc[valid_p95, 'p95']
zero_p95_mask = trace_anomalies['p95'] == 0
trace_anomalies.loc[zero_p95_mask & (trace_anomalies['max_value'] > 0), 'max_over_p95_ratio'] = np.inf

# Keep only groups with anomaly_count > 0 and order/format columns
trace_anomalies = trace_anomalies[trace_anomalies['anomaly_count'] > 0].reset_index()
trace_anomalies = trace_anomalies[['cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anom_ts', 'max_value', 'p95', 'max_over_p95_ratio']]
trace_anomalies = trace_anomalies.sort_values('anomaly_count', ascending=False).reset_index(drop=True)

# Display compact top 20 results and keep trace_anomalies in kernel
trace_anomalies.head(20)

```
Out[3]:
```
```python
# Create a concise plain-English summary of the trace_anomalies results.
# Reuse trace_anomalies from kernel state and output a single summary string.

ta = trace_anomalies  # reuse variable from kernel
total_groups = int(ta.shape[0])

lines = []
lines.append("Trace anomalies summary for 2021-03-10 22:00–22:30 (UTC+8):")
lines.append(f"- Total anomalous (cmdb_id, trace_name) groups: {total_groups}.")

if total_groups == 0:
    lines.append("- No anomalies detected in the specified time window.")
else:
    # Top 5 by anomaly_count
    lines.append("- Top anomaly groups (by anomaly_count):")
    top5 = ta.head(5)
    for _, r in top5.iterrows():
        cmdb = r['cmdb_id']
        name = r['trace_name']
        cnt = int(r['anomaly_count'])
        ts = r['earliest_anom_ts']
        maxv = r['max_value']
        p95 = r['p95']
        ratio = r['max_over_p95_ratio']
        ratio_str = ("NaN" if pd.isna(ratio) else ("inf" if ratio == float('inf') else f"{ratio:.3f}"))
        lines.append(f"  • {cmdb} | {name} | anomalies: {cnt} | earliest: {ts} | max: {maxv:.6g} | p95: {p95:.6g} | max/p95: {ratio_str}")

    # Overall highest ratio
    valid_ratios = ta[ta['max_over_p95_ratio'].notna() & (ta['max_over_p95_ratio'] != float('inf'))]
    if not valid_ratios.empty:
        idx = valid_ratios['max_over_p95_ratio'].idxmax()
        r = ta.loc[idx]
        lines.append(f"- Highest max_over_p95_ratio: {r['max_over_p95_ratio']:.3f} for {r['cmdb_id']} | {r['trace_name']} (earliest: {r['earliest_anom_ts']}).")
    else:
        # check for infinite ratios
        if (ta['max_over_p95_ratio'] == float('inf')).any():
            inf_row = ta[ta['max_over_p95_ratio'] == float('inf')].iloc[0]
            lines.append(f"- At least one group has p95 == 0 and positive max value, producing an infinite max_over_p95_ratio: {inf_row['cmdb_id']} | {inf_row['trace_name']}.")
        else:
            lines.append("- No valid max_over_p95_ratio values available (p95 missing or zero).")

    # Quick interpretation
    lines.append("- Interpretation: Most anomalies are modest (max/p95 in ~1.0–2.4 range).")
    lines.append("  The top group (dockerB2 trace.self.duration_p95) shows the most frequent anomalies (16 minutes within the 30-min window).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                        trace_name  anomaly_count          earliest_anom_ts   max_value         p95  max_over_p95_ratio
0   dockerB2           trace.self.duration_p95             16 2021-03-10 22:00:00+08:00    0.010000    0.009000            1.111111
1   dockerA1         trace.from_MG02.row_count              6 2021-03-10 22:02:00+08:00   59.000000   55.000000            1.072727
2       MG02       trace.to_dockerA1.row_count              6 2021-03-10 22:02:00+08:00   59.000000   55.000000            1.072727
3   dockerA2           trace.self.duration_p95              4 2021-03-10 22:12:00+08:00    0.012500    0.012000            1.041667
4   Tomcat02           trace.to_MG02.row_count              3 2021-03-10 22:01:00+08:00   77.000000   75.000000            1.026667
5   Tomcat02           trace.self.duration_p95              3 2021-03-10 22:08:00+08:00    0.045000    0.044000            1.022727
6   Tomcat04        trace.to_MG01.duration_p95              3 2021-03-10 22:07:00+08:00    1.173000    1.056100            1.110690
7   dockerB1           trace.self.duration_p95              3 2021-03-10 22:07:00+08:00    0.012000    0.012000            1.000000
8       MG02     trace.from_Tomcat02.row_count              3 2021-03-10 22:01:00+08:00   77.000000   75.000000            1.026667
9       MG01  trace.from_Tomcat04.duration_p95              3 2021-03-10 22:07:00+08:00    1.173000    1.056100            1.110690
10  dockerA2          trace.self.duration_mean              2 2021-03-10 22:01:00+08:00    0.031868    0.024952            1.277182
11  dockerB2       trace.to_MG02.duration_mean              2 2021-03-10 22:06:00+08:00    0.082825    0.044233            1.872486
12  dockerA2         trace.from_MG01.row_count              2 2021-03-10 22:03:00+08:00   64.000000   61.650000            1.038118
13  dockerA2     trace.from_MG02.duration_mean              2 2021-03-10 22:01:00+08:00    0.333306    0.267383            1.246551
14  dockerA2       trace.to_MG01.duration_mean              2 2021-03-10 22:01:00+08:00    0.138875    0.093000            1.493280
15  dockerA2              trace.self.row_count              2 2021-03-10 22:03:00+08:00  872.000000  844.150000            1.032992
16  dockerA2           trace.to_MG01.row_count              2 2021-03-10 22:10:00+08:00   27.000000   21.100000            1.279621
17  dockerA2       trace.to_MG02.duration_mean              2 2021-03-10 22:10:00+08:00    0.125789    0.085943            1.463640
18  dockerA2        trace.to_MG02.duration_p95              2 2021-03-10 22:10:00+08:00    1.017000    0.420450            2.418837
19  dockerA2        trace.to_MG01.duration_p95              2 2021-03-10 22:01:00+08:00    1.032750    0.473307            2.181985```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: df_metric, agg_metric, metric_anomalies
# Ensure timestamp_dt exists and is in Asia/Shanghai
if 'timestamp_dt' not in df_metric.columns:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
else:
    df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp_dt']).dt.tz_convert(tz)

# Time window
start = pd.Timestamp('2021-03-10 22:00:00', tz=tz)
end = pd.Timestamp('2021-03-10 22:30:00', tz=tz)

# Groups to consider (from metric_anomalies)
groups_considered = metric_anomalies[['cmdb_id', 'kpi_name']].drop_duplicates()

# Filter metric points to time window and to the groups considered
df_window = df_metric[
    (df_metric['timestamp_dt'] >= start) & (df_metric['timestamp_dt'] <= end)
].merge(
    groups_considered, on=['cmdb_id', 'kpi_name'], how='inner'
).copy()

# Merge global p95 from agg_metric (precomputed global thresholds)
df_window = df_window.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Prepare seconds column for exact-60s adjacency checks
df_window['ts_seconds'] = df_window['timestamp_dt'].astype('int64') // 10**9

# Mark anomaly points where value >= global p95 (only when p95 is present)
df_window['is_anom'] = False
mask_valid = df_window['p95'].notna()
df_window.loc[mask_valid, 'is_anom'] = df_window.loc[mask_valid, 'value'] >= df_window.loc[mask_valid, 'p95']

# Keep only anomaly rows
df_anom = df_window[df_window['is_anom']].copy()

# If no anomaly rows, create empty metric_faults with the requested schema
if df_anom.empty:
    metric_faults = pd.DataFrame(columns=[
        'cmdb_id', 'kpi_name', 'fault_start_ts', 'fault_end_ts',
        'rows_count', 'duration_minutes', 'max_value', 'p95', 'max_over_p95_ratio'
    ])
else:
    # Sort and compute diffs within each (cmdb_id, kpi_name)
    df_anom = df_anom.sort_values(['cmdb_id', 'kpi_name', 'ts_seconds']).reset_index(drop=True)
    df_anom['diff_s'] = df_anom.groupby(['cmdb_id', 'kpi_name'])['ts_seconds'].diff().fillna(60)
    # Start a new segment when diff_s != 60
    df_anom['new_seg'] = df_anom['diff_s'] != 60
    # Segment id per group
    df_anom['seg_id'] = df_anom.groupby(['cmdb_id', 'kpi_name'])['new_seg'].cumsum()

    # Aggregate each segment
    seg_agg = df_anom.groupby(['cmdb_id', 'kpi_name', 'seg_id']).agg(
        fault_start_ts=('timestamp_dt', 'min'),
        fault_end_ts=('timestamp_dt', 'max'),
        rows_count=('timestamp_dt', 'size'),
        max_value=('value', 'max'),
        p95=('p95', 'first')
    ).reset_index(drop=False)

    # Duration in minutes (consecutive minutes => rows_count minutes)
    seg_agg['duration_minutes'] = seg_agg['rows_count'].astype(int)

    # Compute max_over_p95_ratio with proper handling for p95==0 or NaN
    seg_agg['max_over_p95_ratio'] = np.nan
    valid_p95_mask = seg_agg['p95'].notna() & (seg_agg['p95'] != 0)
    seg_agg.loc[valid_p95_mask, 'max_over_p95_ratio'] = seg_agg.loc[valid_p95_mask, 'max_value'] / seg_agg.loc[valid_p95_mask, 'p95']
    zero_p95_mask = seg_agg['p95'] == 0
    seg_agg.loc[zero_p95_mask & (seg_agg['max_value'] > 0), 'max_over_p95_ratio'] = np.inf

    # Filter out short segments (rows_count < 3)
    metric_faults = seg_agg[seg_agg['rows_count'] >= 3].copy()

    # Select and order columns as requested
    metric_faults = metric_faults[[
        'cmdb_id', 'kpi_name', 'fault_start_ts', 'fault_end_ts',
        'rows_count', 'duration_minutes', 'max_value', 'p95', 'max_over_p95_ratio'
    ]]

    # Sort by rows_count desc then max_over_p95_ratio desc
    metric_faults = metric_faults.sort_values(['rows_count', 'max_over_p95_ratio'], ascending=[False, False]).reset_index(drop=True)

# Store metric_faults in kernel (variable already assigned) and display top 20 rows compactly
metric_faults.head(20)

```
Out[4]:
```
Summary of detected metric fault segments (2021-03-10 22:00–22:30 UTC+8):

- Number shown: top 20 fault segments (as returned). Many groups have a single continuous segment spanning the entire 31-minute window (rows_count = 31, duration = 31 minutes).

- Pattern:
  - Most segments (examples: IG01 JVM memory metrics, MG01/MG02 JVM memory metrics, many OSLinux process metrics, ServiceTest app.* metrics, Mysql01/02, Redis01/02) show max_value == p95 and max_over_p95_ratio == 1.0 — i.e., the metric was at or exactly equal to its global p95 for every minute in the window.
  - One notable exception: Mysql02 · OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT has max_value = 2 and p95 = 0, producing an infinite max_over_p95_ratio (p95 == 0). This indicates the global p95 is zero while positive values appeared in the window.

- Interpretation:
  - The many length-31 segments indicate persistent values at or above the global p95 throughout the entire 30‑minute window, not short transient spikes.
  - Where p95 == 0 but values > 0 (infinite ratio), investigate how the global p95 was computed (possibly sparse historical data) and confirm whether nonzero values here are meaningful anomalies.

- Suggested next steps:
  - Inspect raw time series for groups with infinite ratio or sustained p95 equality to determine if thresholds are appropriate or if there is a true sustained fault.
  - For groups with sustained anomalies, check related logs/traces and resource KPIs (CPU, disk, network) to identify possible root causes.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name            fault_start_ts              fault_end_ts  rows_count  duration_minutes     max_value           p95  max_over_p95_ratio
0        Mysql02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.000000e+00  0.000000e+00                 inf
1           IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.040136e+09  2.040136e+09                 1.0
2           IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  5.792518e+07  5.792518e+07                 1.0
3           IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  1.000000e+00  1.000000e+00                 1.0
4           IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P... 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  3.000000e-01  3.000000e-01                 1.0
5           IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  3.000000e+00  3.000000e+00                 1.0
6           IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.040136e+09  2.040136e+09                 1.0
7           IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  1.000000e+00  1.000000e+00                 1.0
8           IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  3.000000e+00  3.000000e+00                 1.0
9           MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.040136e+09  2.040136e+09                 1.0
10          MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  5.069452e+08  5.069452e+08                 1.0
11          MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.040136e+09  2.040136e+09                 1.0
12          MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  9.471521e+08  9.471521e+08                 1.0
13       Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  7.000000e+00  7.000000e+00                 1.0
14       Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  2.400000e+01  2.400000e+01                 1.0
15       Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  1.000000e+00  1.000000e+00                 1.0
16       Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  4.000000e+00  4.000000e+00                 1.0
17       Redis02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  4.000000e+00  4.000000e+00                 1.0
18  ServiceTest1                                             app.rr 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  1.000000e+02  1.000000e+02                 1.0
19  ServiceTest1                                             app.sr 2021-03-10 22:00:00+08:00 2021-03-10 22:30:00+08:00          31                31  1.000000e+02  1.000000e+02                 1.0```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse metric_faults and trace_anomalies from kernel state

# Ensure timestamp columns are timezone-aware and in Asia/Shanghai
if not metric_faults.empty:
    if 'fault_start_ts' in metric_faults.columns:
        # convert/ensure tz-aware
        metric_faults['fault_start_ts'] = pd.to_datetime(metric_faults['fault_start_ts']).dt.tz_convert(tz)
    if 'fault_end_ts' in metric_faults.columns:
        metric_faults['fault_end_ts'] = pd.to_datetime(metric_faults['fault_end_ts']).dt.tz_convert(tz)

if not trace_anomalies.empty:
    if 'earliest_anom_ts' in trace_anomalies.columns:
        trace_anomalies['earliest_anom_ts'] = pd.to_datetime(trace_anomalies['earliest_anom_ts']).dt.tz_convert(tz)

# Aggregate metric_faults by cmdb_id
if metric_faults.empty:
    metric_grp = pd.DataFrame(columns=['cmdb_id', 'earliest_metric_fault_start', 'max_metric_ratio', 'metric_fault_count'])
else:
    metric_grp = metric_faults.groupby('cmdb_id').agg(
        earliest_metric_fault_start=('fault_start_ts', 'min'),
        max_metric_ratio=('max_over_p95_ratio', 'max'),
        metric_fault_count=('cmdb_id', 'size')
    ).reset_index()

# Aggregate trace_anomalies by cmdb_id
if trace_anomalies.empty:
    trace_grp = pd.DataFrame(columns=['cmdb_id', 'earliest_trace_anom_ts', 'max_trace_ratio', 'trace_anom_count'])
else:
    trace_grp = trace_anomalies.groupby('cmdb_id').agg(
        earliest_trace_anom_ts=('earliest_anom_ts', 'min'),
        max_trace_ratio=('max_over_p95_ratio', 'max'),
        trace_anom_count=('cmdb_id', 'size')
    ).reset_index()

# Outer join on cmdb_id to combine
candidate_components = pd.merge(metric_grp, trace_grp, on='cmdb_id', how='outer')

# Ensure columns exist even if one side was empty
for col in ['earliest_metric_fault_start', 'max_metric_ratio', 'metric_fault_count']:
    if col not in candidate_components.columns:
        candidate_components[col] = pd.NaT if 'earliest' in col else np.nan
for col in ['earliest_trace_anom_ts', 'max_trace_ratio', 'trace_anom_count']:
    if col not in candidate_components.columns:
        candidate_components[col] = pd.NaT if 'earliest' in col else np.nan

# Compute overall_earliest as the min of the two timestamp columns ignoring NaT
candidate_components['overall_earliest'] = candidate_components[['earliest_metric_fault_start', 'earliest_trace_anom_ts']].min(axis=1)

# Fill counts NaN with 0 and ensure int dtype
candidate_components['metric_fault_count'] = candidate_components['metric_fault_count'].fillna(0).astype(int)
candidate_components['trace_anom_count'] = candidate_components['trace_anom_count'].fillna(0).astype(int)

# Keep only requested columns and sort by overall_earliest ascending (NaT go last)
candidate_components = candidate_components[[
    'cmdb_id',
    'earliest_metric_fault_start',
    'earliest_trace_anom_ts',
    'overall_earliest',
    'max_metric_ratio',
    'max_trace_ratio',
    'metric_fault_count',
    'trace_anom_count'
]].sort_values('overall_earliest', na_position='last').reset_index(drop=True)

# Store in kernel variable (already assigned) and display compact top 20 rows
candidate_components.head(20)

```
Out[5]:
```
Summary of candidate components (aggregated from metric_faults and trace_anomalies):

- Total components reported: 20 (top candidates shown).

- Components with both metric faults and trace anomalies (high priority):
  - IG01 — earliest metric fault 2021-03-10 22:00, earliest trace anomaly 22:00, overall earliest 22:00; metric_fault_count=8, trace_anom_count=18; max_metric_ratio=1.00, max_trace_ratio≈1.313.
  - Tomcat02 — earliest both 22:00; metric_fault_count=24, trace_anom_count=18; max_metric_ratio=1.00, max_trace_ratio≈1.221.
  - Tomcat01 — earliest both 22:00; metric_fault_count=19, trace_anom_count=18; max_metric_ratio=1.00, max_trace_ratio≈1.434.
  - Tomcat03 & Tomcat04 — earliest both 22:00 (trace earliest 22:01 for some); each metric_fault_count=19, trace_anom_count=18; max_trace_ratio ≈1.30–1.32.

- Components with only trace anomalies (no metric faults in this window):
  - dockerA1 — earliest trace anomaly 22:00; trace_anom_count=18; max_trace_ratio≈1.559 (highest trace ratio among top rows) — investigate traces on dockerA1.
  - dockerA2/dockerB1/dockerB2 and MG01/MG02 appear in trace_anomalies elsewhere but not all are in this top-20 summary.

- Components with only metric faults (no trace anomalies):
  - apache02 — earliest metric fault 22:00; metric_fault_count=15; max_metric_ratio≈1.333.
  - apache01 — earliest metric fault 22:00; metric_fault_count=11.
  - Redis02 / Redis01 — large metric_fault_count (28 and 26) but max_metric_ratio ≈1.0004 and 1.0 (values sit at or barely above p95).
  - Many ServiceTest* entries show brief metric faults (metric_fault_count=2) with max_metric_ratio=1.0.

- Timing and severity patterns:
  - Most overall_earliest timestamps are 2021-03-10 22:00:00 (UTC+8), i.e., anomalies/faults start at the beginning of the 30‑minute window.
  - Several metric segments are persistent through the whole window (rows_count = 31), implying sustained condition rather than transient spikes.
  - max_over_p95_ratio values are modest for most groups (≈1.0–1.6). dockerA1 and a few trace groups have the largest trace ratios (>1.5 and up to ~2.4 in other detailed lists).

- Immediate recommendations:
  1. Prioritize components with both metric faults and trace anomalies (IG01, Tomcat01–04, Tomcat02 especially) for investigation — these show correlated signals across metrics and traces.
  2. Inspect dockerA1 traces (high trace ratio, many anomalies) even though metric faults are not present — possible request/latency issues.
  3. For components with metric faults that equal p95 across the entire window (many groups with ratio ≈1.0), review whether the p95 thresholds are appropriate or if the state is a true sustained issue.
  4. Investigate groups with infinite ratio (p95 == 0 with positive values) separately to verify historical baseline and whether the anomaly is meaningful.

If you want, I can (a) produce a ranked short table of the top 10 components to investigate, (b) show the exact candidate_components rows again, or (c) cross-link these candidates to raw logs/error entries next. Which do you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id earliest_metric_fault_start    earliest_trace_anom_ts          overall_earliest  max_metric_ratio  max_trace_ratio  metric_fault_count  trace_anom_count
0            IG01   2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00          1.000000         1.312819                   8                18
1        dockerA1                         NaT 2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00               NaN         1.559140                   0                18
2        apache02   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.333333              NaN                  15                 0
3        apache01   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                  11                 0
4        Tomcat04   2021-03-10 22:00:00+08:00 2021-03-10 22:01:00+08:00 2021-03-10 22:00:00+08:00          1.000000         1.321107                  19                18
5        Tomcat03   2021-03-10 22:00:00+08:00 2021-03-10 22:01:00+08:00 2021-03-10 22:00:00+08:00          1.000000         1.304285                  19                18
6        Tomcat02   2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00          1.000000         1.220757                  24                18
7        Tomcat01   2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00 2021-03-10 22:00:00+08:00          1.000000         1.433645                  19                18
8    ServiceTest9   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
9    ServiceTest8   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
10   ServiceTest7   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
11   ServiceTest6   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
12   ServiceTest4   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
13   ServiceTest5   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
14   ServiceTest2   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
15  ServiceTest11   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
16  ServiceTest10   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
17   ServiceTest1   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                   2                 0
18        Redis02   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000356              NaN                  28                 0
19        Redis01   2021-03-10 22:00:00+08:00                       NaT 2021-03-10 22:00:00+08:00          1.000000              NaN                  26                 0```
```

In [ ]:
# In[6]:

import re
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Top candidate components to consider
top_candidates = {'IG01', 'Tomcat01', 'Tomcat02', 'Tomcat03', 'Tomcat04',
                  'dockerA1', 'dockerA2', 'dockerB1', 'dockerB2',
                  'MG01', 'MG02', 'apache01', 'apache02'}

# Reuse trace_anomalies and metric_faults from kernel
ta = trace_anomalies.copy() if 'trace_anomalies' in globals() else pd.DataFrame()
mf = metric_faults.copy() if 'metric_faults' in globals() else pd.DataFrame()

# Ensure timestamp columns are timezone-aware in Asia/Shanghai
if not ta.empty and 'earliest_anom_ts' in ta.columns:
    ta['earliest_anom_ts'] = pd.to_datetime(ta['earliest_anom_ts']).dt.tz_convert(tz)

if not mf.empty and 'fault_start_ts' in mf.columns:
    mf['fault_start_ts'] = pd.to_datetime(mf['fault_start_ts']).dt.tz_convert(tz)

# Filter trace anomalies to top candidates and to/from traces
if ta.empty:
    trace_links = pd.DataFrame(columns=[
        'source_cmdb', 'trace_name', 'counterpart_cmdb', 'anomaly_count', 'earliest_anom_ts',
        'max_value', 'p95', 'max_over_p95_ratio', 'counterpart_has_metric_fault', 'counterpart_earliest_metric_fault'
    ])
else:
    ta_f = ta[ta['cmdb_id'].isin(top_candidates)].copy()
    # keep only trace_name containing 'to_' or 'from_'
    ta_f = ta_f[ta_f['trace_name'].str.contains(r'\b(to_|from_)', regex=True, na=False)].copy()
    # extract counterpart using regex
    def extract_counterpart(name):
        m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)', name)
        return m.group(1) if m else None

    ta_f['counterpart_cmdb'] = ta_f['trace_name'].apply(extract_counterpart)
    # drop rows without parsed counterpart or where counterpart is null
    ta_f = ta_f[ta_f['counterpart_cmdb'].notna()].copy()

    # Prepare metric_fault lookup: earliest fault_start_ts per cmdb_id
    if mf.empty:
        mf_agg = pd.DataFrame(columns=['cmdb_id', 'earliest_metric_fault_start'])
    else:
        mf_agg = mf.groupby('cmdb_id', as_index=False).agg(
            earliest_metric_fault_start=('fault_start_ts', 'min')
        )

    # Build trace_links with requested columns
    trace_links = ta_f[['cmdb_id', 'trace_name', 'counterpart_cmdb', 'anomaly_count',
                        'earliest_anom_ts', 'max_value', 'p95', 'max_over_p95_ratio']].copy()
    trace_links = trace_links.rename(columns={'cmdb_id': 'source_cmdb'})

    # Determine counterpart_has_metric_fault and counterpart_earliest_metric_fault by merge
    trace_links = trace_links.merge(
        mf_agg.rename(columns={'cmdb_id': 'counterpart_cmdb', 'earliest_metric_fault_start': 'counterpart_earliest_metric_fault'}),
        on='counterpart_cmdb',
        how='left'
    )

    trace_links['counterpart_has_metric_fault'] = trace_links['counterpart_earliest_metric_fault'].notna()
    # Ensure timezone for counterpart_earliest_metric_fault
    if 'counterpart_earliest_metric_fault' in trace_links.columns:
        trace_links['counterpart_earliest_metric_fault'] = pd.to_datetime(trace_links['counterpart_earliest_metric_fault']).dt.tz_convert(tz)

    # Reorder columns as requested
    trace_links = trace_links[[
        'source_cmdb', 'trace_name', 'counterpart_cmdb', 'anomaly_count', 'earliest_anom_ts',
        'max_value', 'p95', 'max_over_p95_ratio', 'counterpart_has_metric_fault', 'counterpart_earliest_metric_fault'
    ]]

    # Sort by earliest_anom_ts ascending and limit to top 20
    trace_links = trace_links.sort_values('earliest_anom_ts', na_position='last').reset_index(drop=True).head(20)

# Store trace_links in kernel and display compact result
trace_links.head(20)

```
Out[6]:
```
```python
# Compose a concise plain-English summary of trace_links findings.
# Reuse trace_links from kernel state and produce one summary string.

import pandas as pd

tl = trace_links.copy() if 'trace_links' in globals() else pd.DataFrame()

if tl.empty:
    summary = "No trace link anomalies found for the selected candidate components."
else:
    total = int(len(tl))
    earliest = tl['earliest_anom_ts'].min()
    # counts of counterparts with/without metric faults
    with_fault = int(tl['counterpart_has_metric_fault'].sum())
    without_fault = int((~tl['counterpart_has_metric_fault']).sum())
    # top 5 earliest anomalies
    top5 = tl.sort_values('earliest_anom_ts').head(5)[[
        'source_cmdb','trace_name','counterpart_cmdb','earliest_anom_ts','max_over_p95_ratio','counterpart_has_metric_fault'
    ]].to_dict(orient='records')
    # top 3 highest trace ratios
    top_ratios = tl.sort_values('max_over_p95_ratio', ascending=False).head(3)[[
        'source_cmdb','trace_name','counterpart_cmdb','earliest_anom_ts','max_over_p95_ratio','counterpart_has_metric_fault'
    ]].to_dict(orient='records')
    # counterparts without metric faults, ranked by frequency
    no_fault_counts = (tl[~tl['counterpart_has_metric_fault']]
                       .groupby('counterpart_cmdb').size().reset_index(name='count')
                       .sort_values('count', ascending=False))
    no_fault_list = no_fault_counts.to_dict(orient='records')

    lines = []
    lines.append(f"Trace-to/from link anomalies involving top candidates: {total} rows found.")
    lines.append(f"Earliest trace anomaly timestamp: {earliest}.")
    lines.append(f"Counterpart metric-fault presence: {with_fault} links point to counterparts that have metric faults; {without_fault} links point to counterparts without metric faults.")
    lines.append("")
    lines.append("Top 5 earliest trace anomalies:")
    for r in top5:
        lines.append(f" - {r['earliest_anom_ts']}: {r['source_cmdb']} -> {r['trace_name']} -> counterpart {r['counterpart_cmdb']}; ratio={r['max_over_p95_ratio']:.3f}; counterpart_has_metric_fault={bool(r['counterpart_has_metric_fault'])}")
    lines.append("")
    lines.append("Top 3 highest trace max_over_p95_ratio:")
    for r in top_ratios:
        lines.append(f" - {r['earliest_anom_ts']}: {r['source_cmdb']} -> {r['trace_name']} -> counterpart {r['counterpart_cmdb']}; ratio={r['max_over_p95_ratio']:.3f}; counterpart_has_metric_fault={bool(r['counterpart_has_metric_fault'])}")
    lines.append("")
    if no_fault_list:
        lines.append("Counterparts with trace anomalies but no metric faults (by frequency):")
        for rec in no_fault_list:
            lines.append(f" - {rec['counterpart_cmdb']}: {rec['count']} trace links")
    else:
        lines.append("All counterpart components referenced by these traces have metric faults in the metric_faults table.")

    lines.append("")
    lines.append("Interpretation / next steps:")
    lines.append(" - Many trace anomalies map to counterparts that also show metric faults (correlated signals).")
    lines.append(" - Notable standalone cases: counterparts like dockerA1, dockerB1, dockerB2 (appear here) have trace-side anomalies but lack metric faults; investigate those endpoints for latency or routing issues.")
    lines.append(" - Prioritize investigation on links with both high anomaly counts and high max_over_p95_ratio, and on links where counterpart has metric faults for potential cascading issues.")

    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

source_cmdb                         trace_name counterpart_cmdb  anomaly_count          earliest_anom_ts   max_value        p95  max_over_p95_ratio  counterpart_has_metric_fault counterpart_earliest_metric_fault
0     Tomcat01          trace.from_IG02.row_count             IG02              2 2021-03-10 22:00:00+08:00   80.000000  75.550000            1.058901                          True         2021-03-10 22:00:00+08:00
1         MG01      trace.from_Tomcat02.row_count         Tomcat02              2 2021-03-10 22:00:00+08:00   84.000000  78.550000            1.069383                          True         2021-03-10 22:00:00+08:00
2         MG02      trace.from_dockerA1.row_count         dockerA1              2 2021-03-10 22:00:00+08:00   29.000000  18.600000            1.559140                         False                               NaT
3     Tomcat02            trace.to_MG01.row_count             MG01              2 2021-03-10 22:00:00+08:00   84.000000  78.550000            1.069383                          True         2021-03-10 22:00:00+08:00
4     dockerA1            trace.to_MG02.row_count             MG02              2 2021-03-10 22:00:00+08:00   29.000000  18.600000            1.559140                          True         2021-03-10 22:00:00+08:00
5         MG02        trace.to_dockerB1.row_count         dockerB1              2 2021-03-10 22:01:00+08:00   75.000000  67.650000            1.108647                         False                               NaT
6     Tomcat04            trace.to_MG02.row_count             MG02              2 2021-03-10 22:01:00+08:00   83.000000  75.650000            1.097158                          True         2021-03-10 22:00:00+08:00
7         MG01  trace.from_Tomcat01.duration_mean         Tomcat01              2 2021-03-10 22:01:00+08:00    0.194243   0.183974            1.055816                          True         2021-03-10 22:00:00+08:00
8     Tomcat04            trace.to_MG01.row_count             MG01              2 2021-03-10 22:01:00+08:00   82.000000  81.550000            1.005518                          True         2021-03-10 22:00:00+08:00
9     Tomcat01         trace.to_MG02.duration_p95             MG02              2 2021-03-10 22:01:00+08:00    1.192600   1.150325            1.036750                          True         2021-03-10 22:00:00+08:00
10        MG01      trace.from_Tomcat01.row_count         Tomcat01              2 2021-03-10 22:01:00+08:00   82.000000  79.100000            1.036662                          True         2021-03-10 22:00:00+08:00
11    dockerB2            trace.to_MG02.row_count             MG02              2 2021-03-10 22:01:00+08:00  109.000000  90.050000            1.210439                          True         2021-03-10 22:00:00+08:00
12    Tomcat01        trace.to_MG02.duration_mean             MG02              2 2021-03-10 22:01:00+08:00    0.302443   0.244909            1.234917                          True         2021-03-10 22:00:00+08:00
13    Tomcat01            trace.to_MG01.row_count             MG01              2 2021-03-10 22:01:00+08:00   82.000000  79.100000            1.036662                          True         2021-03-10 22:00:00+08:00
14        MG01        trace.to_dockerB1.row_count         dockerB1              2 2021-03-10 22:01:00+08:00   96.000000  76.950000            1.247563                         False                               NaT
15    Tomcat02          trace.from_IG02.row_count             IG02              2 2021-03-10 22:01:00+08:00   87.000000  83.200000            1.045673                          True         2021-03-10 22:00:00+08:00
16    Tomcat01        trace.to_MG01.duration_mean             MG01              2 2021-03-10 22:01:00+08:00    0.194243   0.183974            1.055816                          True         2021-03-10 22:00:00+08:00
17    Tomcat01       trace.from_IG01.duration_p95             IG01              2 2021-03-10 22:01:00+08:00    1.286400   1.146240            1.122278                          True         2021-03-10 22:00:00+08:00
18        MG02        trace.to_dockerB2.row_count         dockerB2              2 2021-03-10 22:01:00+08:00   97.000000  79.400000            1.221662                         False                               NaT
19        MG01     trace.to_dockerB1.duration_p95         dockerB1              2 2021-03-10 22:01:00+08:00    1.247750   1.210457            1.030809                         False                               NaT```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: df_metric, df_trace, agg_metric, agg_trace
# Ensure timestamp_dt exists and is in Asia/Shanghai
for df in (df_metric, df_trace):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt']).dt.tz_convert(tz)

# Time window (inclusive)
start = pd.Timestamp('2021-03-10 22:00:00', tz=tz)
end = pd.Timestamp('2021-03-10 22:30:00', tz=tz)

# 1) Metric per-minute anomaly counts
# Filter metric rows to window
df_metric_window = df_metric[(df_metric['timestamp_dt'] >= start) & (df_metric['timestamp_dt'] <= end)].copy()

# Merge global p95 from agg_metric
df_metric_merge = df_metric_window.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Determine anomaly rows where p95 is present and value >= p95
df_metric_merge['is_anom'] = False
valid_mask_m = df_metric_merge['p95'].notna()
df_metric_merge.loc[valid_mask_m, 'is_anom'] = df_metric_merge.loc[valid_mask_m, 'value'] >= df_metric_merge.loc[valid_mask_m, 'p95']

# Count anomalies per timestamp
metric_anomaly_times = (
    df_metric_merge.groupby('timestamp_dt', as_index=False)
    .agg(metric_anomaly_count=('is_anom', 'sum'))
)

# Ensure timestamp column named 'timestamp' for output
metric_anomaly_times = metric_anomaly_times.rename(columns={'timestamp_dt': 'timestamp'})

# Top 10 timestamps by metric_anomaly_count desc
metric_top10 = metric_anomaly_times.sort_values('metric_anomaly_count', ascending=False).head(10).reset_index(drop=True)

# 2) Trace per-minute anomaly counts
df_trace_window = df_trace[(df_trace['timestamp_dt'] >= start) & (df_trace['timestamp_dt'] <= end)].copy()

df_trace_merge = df_trace_window.merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

df_trace_merge['is_anom'] = False
valid_mask_t = df_trace_merge['p95'].notna()
df_trace_merge.loc[valid_mask_t, 'is_anom'] = df_trace_merge.loc[valid_mask_t, 'value'] >= df_trace_merge.loc[valid_mask_t, 'p95']

trace_anomaly_times = (
    df_trace_merge.groupby('timestamp_dt', as_index=False)
    .agg(trace_anomaly_count=('is_anom', 'sum'))
).rename(columns={'timestamp_dt': 'timestamp'})

trace_top10 = trace_anomaly_times.sort_values('trace_anomaly_count', ascending=False).head(10).reset_index(drop=True)

# 3) Merged top-10 by combined_anomaly_count
# Merge metric and trace per timestamp (outer join), fill missing with 0
merged = pd.merge(metric_anomaly_times.rename(columns={'timestamp':'timestamp'}),
                  trace_anomaly_times.rename(columns={'timestamp':'timestamp'}),
                  on='timestamp', how='outer').fillna(0)

# Ensure integer counts
merged['metric_anomaly_count'] = merged['metric_anomaly_count'].astype(int)
merged['trace_anomaly_count'] = merged['trace_anomaly_count'].astype(int)

merged['combined_anomaly_count'] = merged['metric_anomaly_count'] + merged['trace_anomaly_count']

merged_top10 = merged.sort_values('combined_anomaly_count', ascending=False).head(10).reset_index(drop=True)

# 4) For top 2 timestamps by combined_anomaly_count, find top 5 components by combined anomaly rows
top2_timestamps = merged_top10['timestamp'].head(2).tolist()

rows_list = []
for ts in top2_timestamps:
    # metrics: count anomaly rows per cmdb_id at this timestamp
    m_mask = (df_metric_merge['timestamp_dt'] == ts) & (df_metric_merge['is_anom'])
    m_counts = df_metric_merge.loc[m_mask].groupby('cmdb_id').size().rename('metric_anomaly_rows').reset_index()
    # traces: count anomaly rows per cmdb_id at this timestamp
    t_mask = (df_trace_merge['timestamp_dt'] == ts) & (df_trace_merge['is_anom'])
    t_counts = df_trace_merge.loc[t_mask].groupby('cmdb_id').size().rename('trace_anomaly_rows').reset_index()
    # Combine per cmdb_id
    combined_ct = pd.merge(m_counts, t_counts, on='cmdb_id', how='outer').fillna(0)
    combined_ct['metric_anomaly_rows'] = combined_ct['metric_anomaly_rows'].astype(int)
    combined_ct['trace_anomaly_rows'] = combined_ct['trace_anomaly_rows'].astype(int)
    combined_ct['total_anomaly_rows'] = combined_ct['metric_anomaly_rows'] + combined_ct['trace_anomaly_rows']
    combined_ct['timestamp'] = ts
    # select top 5 components by total_anomaly_rows desc (tie-breaker: metric_anomaly_rows desc)
    combined_ct = combined_ct.sort_values(['total_anomaly_rows', 'metric_anomaly_rows'], ascending=[False, False]).head(5)
    # reorder columns
    combined_ct = combined_ct[['timestamp', 'cmdb_id', 'metric_anomaly_rows', 'trace_anomaly_rows', 'total_anomaly_rows']]
    rows_list.append(combined_ct)

# Concatenate results for top2 timestamps (may be empty)
if rows_list:
    top_components_top2 = pd.concat(rows_list, ignore_index=True)
else:
    top_components_top2 = pd.DataFrame(columns=['timestamp','cmdb_id','metric_anomaly_rows','trace_anomaly_rows','total_anomaly_rows'])

# Final compact outputs (stored variables and top summaries)
metric_anomaly_times, metric_top10, trace_anomaly_times, trace_top10, merged_top10, top_components_top2

```
Out[7]:
```
Summary of per-minute anomaly counts and top affected components (2021-03-10 22:00–22:30, UTC+8):

1) Metric anomaly hotspots (top 10 minutes by metric anomaly count)
- 2021-03-10 22:02 — 621 metric anomalies
- 2021-03-10 22:26 — 614
- 2021-03-10 22:20 — 586
- 2021-03-10 22:11 — 584
- 2021-03-10 22:14 — 579
- 2021-03-10 22:05 — 575
- 2021-03-10 22:08 — 569
- 2021-03-10 22:17 — 569
- 2021-03-10 22:09 — 565
- 2021-03-10 22:28 — 544

2) Trace anomaly hotspots (top 10 minutes by trace anomaly count)
- 2021-03-10 22:01 — 79 trace anomalies
- 2021-03-10 22:23 — 52
- 2021-03-10 22:10 — 46
- 2021-03-10 22:07 — 37
- 2021-03-10 22:13 — 36
- 2021-03-10 22:20 — 33
- 2021-03-10 22:11 — 22
- 2021-03-10 22:29 — 22
- 2021-03-10 22:08 — 21
- 2021-03-10 22:02 — 19

3) Combined metric+trace hotspots (top 10 minutes by combined_anomaly_count)
- 2021-03-10 22:02 — combined 640 (621 metric + 19 trace)
- 2021-03-10 22:20 — combined 619 (586 metric + 33 trace)
- 2021-03-10 22:26 — combined 618 (614 + 4)
- 2021-03-10 22:11 — combined 606 (584 + 22)
- 2021-03-10 22:08 — combined 590 (569 + 21)
- 2021-03-10 22:14 — combined 589 (579 + 10)
- 2021-03-10 22:05 — combined 585 (575 + 10)
- 2021-03-10 22:17 — combined 582 (569 + 13)
- 2021-03-10 22:23 — combined 575 (523 + 52)
- 2021-03-10 22:09 — combined 565 (565 + 0)

4) Top components at the top 2 combined timestamps (top 5 components per timestamp)
- For 2021-03-10 22:02:
  - Mysql02: 116 metric anomaly rows (0 trace) — total 116
  - Mysql01: 103 metric rows — total 103
  - Redis02: 65 metric rows — total 65
  - Redis01: 55 metric rows — total 55
  - Tomcat03: 50 metric rows — total 50

- For 2021-03-10 22:20:
  - Mysql01: 100 metric rows — total 100
  - Mysql02: 95 metric rows — total 95
  - Tomcat03: 49 metric + 3 trace = total 52
  - MG02: 44 metric + 3 trace = total 47
  - Redis01: 43 metric rows — total 43

Interpretation and recommended next steps
- Metric anomalies dominate in count and are widespread across many components; trace anomalies are fewer but spike at different minutes (notably 22:01 and 22:23).
- The single largest combined anomaly minute is 22:02 (mostly metric-related), followed by 22:20 which shows more trace involvement.
- MySQL instances (Mysql01, Mysql02) and Redis instances (Redis01, Redis02) show the largest per-component metric anomaly volumes at peak minutes — they should be prioritized for investigation.
- Tomcat03 and MG02 show mixed metric and trace anomalies around 22:20, suggesting possible correlated service and request/trace issues.
- Suggested actions: inspect raw time series and logs for Mysql01/02, Redis01/02, Tomcat03, and MG02 around 22:00–22:30; correlate with traces at 22:01, 22:02, 22:20, and 22:23 to find root causes (resource exhaustion, GC, network, or request spikes).

The original code execution output of IPython Kernel is also provided below for reference:

(                   timestamp  metric_anomaly_count
0  2021-03-10 22:00:00+08:00                   485
1  2021-03-10 22:01:00+08:00                   481
2  2021-03-10 22:02:00+08:00                   621
3  2021-03-10 22:03:00+08:00                   523
4  2021-03-10 22:04:00+08:00                   371
..                       ...                   ...
26 2021-03-10 22:26:00+08:00                   614
27 2021-03-10 22:27:00+08:00                   485
28 2021-03-10 22:28:00+08:00                   544
29 2021-03-10 22:29:00+08:00                   536
30 2021-03-10 22:30:00+08:00                   516

[31 rows x 2 columns],                   timestamp  metric_anomaly_count
0 2021-03-10 22:02:00+08:00                   621
1 2021-03-10 22:26:00+08:00                   614
2 2021-03-10 22:20:00+08:00                   586
3 2021-03-10 22:11:00+08:00                   584
4 2021-03-10 22:14:00+08:00                   579
5 2021-03-10 22:05:00+08:00                   575
6 2021-03-10 22:08:00+08:00                   569
7 2021-03-10 22:17:00+08:00                   569
8 2021-03-10 22:09:00+08:00                   565
9 2021-03-10 22:28:00+08:00                   544,                    timestamp  trace_anomaly_count
0  2021-03-10 22:00:00+08:00                    9
1  2021-03-10 22:01:00+08:00                   79
2  2021-03-10 22:02:00+08:00                   19
3  2021-03-10 22:03:00+08:00                   14
4  2021-03-10 22:04:00+08:00                    1
..                       ...                  ...
25 2021-03-10 22:25:00+08:00                   11
26 2021-03-10 22:26:00+08:00                    4
27 2021-03-10 22:27:00+08:00                    1
28 2021-03-10 22:28:00+08:00                    4
29 2021-03-10 22:29:00+08:00                   22

[30 rows x 2 columns],                   timestamp  trace_anomaly_count
0 2021-03-10 22:01:00+08:00                   79
1 2021-03-10 22:23:00+08:00                   52
2 2021-03-10 22:10:00+08:00                   46
3 2021-03-10 22:07:00+08:00                   37
4 2021-03-10 22:13:00+08:00                   36
5 2021-03-10 22:20:00+08:00                   33
6 2021-03-10 22:11:00+08:00                   22
7 2021-03-10 22:29:00+08:00                   22
8 2021-03-10 22:08:00+08:00                   21
9 2021-03-10 22:02:00+08:00                   19,                   timestamp  metric_anomaly_count  trace_anomaly_count  combined_anomaly_count
0 2021-03-10 22:02:00+08:00                   621                   19                     640
1 2021-03-10 22:20:00+08:00                   586                   33                     619
2 2021-03-10 22:26:00+08:00                   614                    4                     618
3 2021-03-10 22:11:00+08:00                   584                   22                     606
4 2021-03-10 22:08:00+08:00                   569                   21                     590
5 2021-03-10 22:14:00+08:00                   579                   10                     589
6 2021-03-10 22:05:00+08:00                   575                   10                     585
7 2021-03-10 22:17:00+08:00                   569                   13                     582
8 2021-03-10 22:23:00+08:00                   523                   52                     575
9 2021-03-10 22:09:00+08:00                   565                    0                     565,                   timestamp   cmdb_id  metric_anomaly_rows  trace_anomaly_rows  total_anomaly_rows
0 2021-03-10 22:02:00+08:00   Mysql02                  116                   0                 116
1 2021-03-10 22:02:00+08:00   Mysql01                  103                   0                 103
2 2021-03-10 22:02:00+08:00   Redis02                   65                   0                  65
3 2021-03-10 22:02:00+08:00   Redis01                   55                   0                  55
4 2021-03-10 22:02:00+08:00  Tomcat03                   50                   0                  50
5 2021-03-10 22:20:00+08:00   Mysql01                  100                   0                 100
6 2021-03-10 22:20:00+08:00   Mysql02                   95                   0                  95
7 2021-03-10 22:20:00+08:00  Tomcat03                   49                   3                  52
8 2021-03-10 22:20:00+08:00      MG02                   44                   3                  47
9 2021-03-10 22:20:00+08:00   Redis01                   43                   0                  43)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import re
import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: df_metric, df_trace, df_log, agg_metric, agg_trace
# Ensure timestamp_dt columns exist and are in Asia/Shanghai
for df in (df_metric, df_trace, df_log):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt']).dt.tz_convert(tz)

# Helper: compute per-timestamp metric details
def build_metric_details_for(ts):
    # Filter metric rows at exact timestamp
    df_m_ts = df_metric[df_metric['timestamp_dt'] == ts].copy()
    # Join global p95
    df_m_ts = df_m_ts.merge(agg_metric[['cmdb_id', 'kpi_name', 'p95']], on=['cmdb_id', 'kpi_name'], how='left')
    # Keep rows where p95 exists and value >= p95
    df_m_ts = df_m_ts[df_m_ts['p95'].notna()].copy()
    df_m_ts['is_anom'] = df_m_ts['value'] >= df_m_ts['p95']
    df_m_ts = df_m_ts[df_m_ts['is_anom']].copy()
    if df_m_ts.empty:
        return pd.DataFrame(columns=['timestamp','cmdb_id','anomaly_count','top_kpis'])
    # Compute ratio per row and then aggregate per (cmdb_id, kpi_name)
    df_m_ts['ratio'] = np.nan
    valid = df_m_ts['p95'] != 0
    df_m_ts.loc[valid, 'ratio'] = df_m_ts.loc[valid, 'value'] / df_m_ts.loc[valid, 'p95']
    zero_mask = (df_m_ts['p95'] == 0) & (df_m_ts['value'] > 0)
    df_m_ts.loc[zero_mask, 'ratio'] = np.inf

    # Aggregate per cmdb_id & kpi_name to get max_value and ratio (though single timestamp, keep robust)
    kpi_agg = df_m_ts.groupby(['cmdb_id', 'kpi_name'], as_index=False).agg(
        max_value=('value', 'max'),
        p95=('p95', 'first'),
        ratio=('ratio', 'max')
    )

    # For each cmdb_id, compute anomaly_count (# of KPI rows) and top 3 KPIs by ratio
    def top_kpis_for_group(g):
        top = g.sort_values('ratio', ascending=False).head(3)
        entries = []
        for _, r in top.iterrows():
            ratio_str = ("inf" if np.isinf(r['ratio']) else (f"{r['ratio']:.3f}" if not pd.isna(r['ratio']) else "NaN"))
            entries.append(f"{r['kpi_name']}|max={r['max_value']:.6g}|p95={r['p95']:.6g}|ratio={ratio_str}")
        return "; ".join(entries)

    grp = kpi_agg.groupby('cmdb_id').agg(
        anomaly_count=('kpi_name', 'nunique'),  # number of distinct KPIs breaching
    ).reset_index()

    topk = kpi_agg.groupby('cmdb_id').apply(top_kpis_for_group).reset_index().rename(columns={0:'top_kpis'})
    metric_details = grp.merge(topk, on='cmdb_id', how='left')
    metric_details.insert(0, 'timestamp', ts)
    # Keep requested columns
    metric_details = metric_details[['timestamp', 'cmdb_id', 'anomaly_count', 'top_kpis']]
    return metric_details

# Helper: compute per-timestamp trace details
def extract_counterpart(name):
    m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)', name)
    return m.group(1) if m else None

def build_trace_details_for(ts):
    df_t_ts = df_trace[df_trace['timestamp_dt'] == ts].copy()
    df_t_ts = df_t_ts.merge(agg_trace[['cmdb_id', 'trace_name', 'p95']], on=['cmdb_id', 'trace_name'], how='left')
    df_t_ts = df_t_ts[df_t_ts['p95'].notna()].copy()
    df_t_ts['is_anom'] = df_t_ts['value'] >= df_t_ts['p95']
    df_t_ts = df_t_ts[df_t_ts['is_anom']].copy()
    if df_t_ts.empty:
        return pd.DataFrame(columns=['timestamp','source_cmdb','counterpart_cmdb','trace_anom_count','max_over_p95_ratio'])
    # parse counterpart
    df_t_ts['counterpart_cmdb'] = df_t_ts['trace_name'].apply(extract_counterpart)
    df_t_ts = df_t_ts[df_t_ts['counterpart_cmdb'].notna()].copy()
    if df_t_ts.empty:
        return pd.DataFrame(columns=['timestamp','source_cmdb','counterpart_cmdb','trace_anom_count','max_over_p95_ratio'])
    # compute ratio
    df_t_ts['ratio'] = np.nan
    valid = df_t_ts['p95'] != 0
    df_t_ts.loc[valid, 'ratio'] = df_t_ts.loc[valid, 'value'] / df_t_ts.loc[valid, 'p95']
    zero_mask = (df_t_ts['p95'] == 0) & (df_t_ts['value'] > 0)
    df_t_ts.loc[zero_mask, 'ratio'] = np.inf

    # aggregate by source cmdb_id and counterpart
    agg = df_t_ts.groupby(['cmdb_id', 'counterpart_cmdb'], as_index=False).agg(
        trace_anom_count=('trace_name', 'size'),
        max_over_p95_ratio=('ratio', 'max')
    ).rename(columns={'cmdb_id':'source_cmdb'})

    agg.insert(0, 'timestamp', ts)
    # sort by trace_anom_count desc and limit top 10
    agg = agg.sort_values(['trace_anom_count', 'max_over_p95_ratio'], ascending=[False, False]).reset_index(drop=True)
    return agg[['timestamp','source_cmdb','counterpart_cmdb','trace_anom_count','max_over_p95_ratio']]

# Helper: get logs for timestamp and cmdb_id set
def build_log_details_for(ts, cmdb_set):
    df_l = df_log[(df_log['timestamp_dt'] == ts) & (df_log['cmdb_id'].isin(cmdb_set))].copy()
    if df_l.empty:
        return pd.DataFrame(columns=['timestamp','cmdb_id','log_name','value'])
    df_l = df_l[['timestamp_dt','cmdb_id','log_name','value']].rename(columns={'timestamp_dt':'timestamp'})
    return df_l

# Define target timestamps
ts1 = pd.Timestamp('2021-03-10 22:02:00', tz=tz)
ts2 = pd.Timestamp('2021-03-10 22:20:00', tz=tz)

# Build metric details
metric_details_2202 = build_metric_details_for(ts1)
metric_details_2220 = build_metric_details_for(ts2)

# Build trace details
trace_details_2202 = build_trace_details_for(ts1)
trace_details_2220 = build_trace_details_for(ts2)

# Build log details: union of cmdb_ids appearing in metric_details and trace_details for each timestamp
cmdbs_2202 = set(metric_details_2202['cmdb_id'].tolist()) | set(trace_details_2202['source_cmdb'].tolist()) | set(trace_details_2202['counterpart_cmdb'].tolist())
cmdbs_2220 = set(metric_details_2220['cmdb_id'].tolist()) | set(trace_details_2220['source_cmdb'].tolist()) | set(trace_details_2220['counterpart_cmdb'].tolist())

log_details_2202 = build_log_details_for(ts1, cmdbs_2202)
log_details_2220 = build_log_details_for(ts2, cmdbs_2220)

# For compact display, show only top rows as requested while storing full tables in kernel variables
metric_details_2202_display = metric_details_2202.head(10)
trace_details_2202_display = trace_details_2202.head(10)
log_details_2202_display = log_details_2202.head(20)

metric_details_2220_display = metric_details_2220.head(10)
trace_details_2220_display = trace_details_2220.head(10)
log_details_2220_display = log_details_2220.head(20)

# Return the six display frames (full tables are stored in variables metric_details_2202, etc.)
metric_details_2202_display, trace_details_2202_display, log_details_2202_display, metric_details_2220_display, trace_details_2220_display, log_details_2220_display

```
Out[8]:
```
Summary of findings for the two peak minutes (stored in kernel variables as requested):

General: all timestamps are UTC+8. The following kernel variables were created and saved:
- For 22:02: metric_details_2202, trace_details_2202, log_details_2202
- For 22:20: metric_details_2220, trace_details_2220, log_details_2220

A. Minute = 2021-03-10 22:02:00+08:00
1) Metric summary (metric_details_2202)
- Top affected components (anomaly_count = number of KPI breaches at that minute): Mysql02 (116), Mysql01 (103), Redis02 (65), Redis01 (55), IG02 (34), IG01 (31), MG01 (14), MG02 (12), plus ServiceTest entries.
- Representative top KPIs (examples from the top_kpis field): InnoDB data written and other MySQL I/O metrics (Mysql02/Mysql01), local disk read metrics, memory/cache KPIs, CPU metrics for MG01. Many breaches indicate I/O / memory / resource KPIs at or above global p95.

2) Trace summary (trace_details_2202)
- Notable trace pairs (source ↔ counterpart) include MG02 ↔ dockerA1 (2 anomalies, ratio ≈1.09), MG02 ↔ dockerA2, MG02 ↔ dockerB2, IG01 ↔ Tomcat04, IG01 ↔ Tomcat02, Tomcat02 ↔ IG01. Ratios are modest (~1.03–1.09).
- Some counterparts (e.g., dockerA1) appear in trace anomalies but do not have metric faults at this minute — possible request/latency issues on the tracing path rather than local resource saturation.

3) Logs at that minute (log_details_2202)
- For the union of involved components, logs show high row_count values for Tomcat and Apache processes (row_count numbers present) and log.error_count = 0 for these servers.
- Example: Tomcat01 row_count ~127, Tomcat02 row_count ~154 and gc_pause_max ~0.07; apache01/02 row_count high and error_count 0.
- No immediate error_count spikes; tracing and metric signals are stronger indicators.

B. Minute = 2021-03-10 22:20:00+08:00
1) Metric summary (metric_details_2220)
- Top affected components: MG02 (44), MG01 (35), Mysql01 (100), Mysql02 (95), IG02 (33), IG01 (19), Redis01 (43), Redis02 (40).
- Representative KPIs: local disk / disk I/O KPIs (MG02), JVM memory/use and heap metrics (MG01/IG02), MySQL binlog/stmt/cache & CPU load (Mysql01/02), filesystem metrics for Redis.
- Many breaches point to disk I/O, JVM memory, and MySQL activity at or above p95.

2) Trace summary (trace_details_2220)
- Multiple source components show trace anomalies pointing to counterpart "root" (each with trace_anom_count = 3 and ratio ≈1.00), including Tomcat03, IG01, IG02, MG01, MG02, Tomcat01/02/04, dockerA1/A2. (Parsing produced 'root' as the counterpart string for these trace names.)
- Ratios are near 1.0 (modest), counts are small but consistent across many sources — suggests a common target/endpoint or naming quirk; investigate trace naming that maps to "root".

3) Logs at that minute (log_details_2220)
- Tomcat and Apache log summaries again show high row_count values and log.error_count = 0 (examples: Tomcat01 row_count 95, Tomcat02 row_count 122, apache01 row_count 217).
- No bulk error log lines at these timestamps.

C. Interpretation and recommended next steps (concise)
- Metric anomalies are dominated by MySQL and Redis instances (Mysql01/02, Redis01/02) at the top minutes — prioritize storage/I/O and DB-level investigation (InnoDB writes, binlog behavior, CPU/load).
- MG/IG and Tomcat nodes show correlated metric and trace signals — investigate JVM memory and disk I/O on those hosts.
- Trace anomalies highlight MG02 ↔ dockerA* links (notably dockerA1) and several Tomcat↔IG links; dockerA1 had trace anomalies but no metric faults — investigate networking/latency or request routing on that path.
- Logs show high traffic (row_count) but no error_count spikes; focus on resource metrics (disk I/O, JVM heap, CPU) rather than application errors initially.
- The trace counterpart "root" found at 22:20 should be inspected (possible parsing artifact or a real endpoint), since many sources reference it.

If you want, I can:
- expand any metric_details_* row to show the full top_kpis strings,
- list raw metric/trace rows for a specific component at 22:02 or 22:20,
- or correlate these findings with metric_faults/trace_anomalies timeline for deeper root-cause drilling. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(                  timestamp        cmdb_id  anomaly_count                                           top_kpis
0 2021-03-10 22:02:00+08:00           IG01             31  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
1 2021-03-10 22:02:00+08:00           IG02             34  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem|max=543...
2 2021-03-10 22:02:00+08:00           MG01             14  OSLinux-CPU_CPU_CPUCpuUtil|max=26.9843|p95=26....
3 2021-03-10 22:02:00+08:00           MG02             12  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
4 2021-03-10 22:02:00+08:00        Mysql01            103  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
5 2021-03-10 22:02:00+08:00        Mysql02            116  Mysql-MySQL_3306_Innodb data written|max=29588...
6 2021-03-10 22:02:00+08:00        Redis01             55  OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace|...
7 2021-03-10 22:02:00+08:00        Redis02             65  OSLinux-CPU_CPU_CPUSysTime|max=0.645699|p95=0....
8 2021-03-10 22:02:00+08:00   ServiceTest1              2  app.rr|max=100|p95=100|ratio=1.000; app.sr|max...
9 2021-03-10 22:02:00+08:00  ServiceTest10              3  app.mrt|max=0.10646|p95=0.095875|ratio=1.110; ...,                   timestamp source_cmdb counterpart_cmdb  trace_anom_count  max_over_p95_ratio
0 2021-03-10 22:02:00+08:00        MG02         dockerA1                 2            1.090652
1 2021-03-10 22:02:00+08:00    dockerA1             MG02                 2            1.090652
2 2021-03-10 22:02:00+08:00        MG02         dockerA2                 1            1.085020
3 2021-03-10 22:02:00+08:00    dockerA2             MG02                 1            1.085020
4 2021-03-10 22:02:00+08:00        MG02         dockerB2                 1            1.063513
5 2021-03-10 22:02:00+08:00    dockerB2             MG02                 1            1.063513
6 2021-03-10 22:02:00+08:00        IG01         Tomcat04                 1            1.055901
7 2021-03-10 22:02:00+08:00    Tomcat04             IG01                 1            1.055901
8 2021-03-10 22:02:00+08:00        IG01         Tomcat02                 1            1.035857
9 2021-03-10 22:02:00+08:00    Tomcat02             IG01                 1            1.035857,                    timestamp   cmdb_id          log_name   value
32 2021-03-10 22:02:00+08:00  Tomcat01   log.error_count    0.00
33 2021-03-10 22:02:00+08:00  Tomcat01     log.row_count  127.00
34 2021-03-10 22:02:00+08:00  Tomcat02   log.error_count    0.00
35 2021-03-10 22:02:00+08:00  Tomcat02  log.gc_pause_max    0.07
36 2021-03-10 22:02:00+08:00  Tomcat02     log.row_count  154.00
37 2021-03-10 22:02:00+08:00  Tomcat03   log.error_count    0.00
38 2021-03-10 22:02:00+08:00  Tomcat03     log.row_count  125.00
39 2021-03-10 22:02:00+08:00  Tomcat04   log.error_count    0.00
40 2021-03-10 22:02:00+08:00  Tomcat04     log.row_count  153.00
41 2021-03-10 22:02:00+08:00  apache01   log.error_count    0.00
42 2021-03-10 22:02:00+08:00  apache01     log.row_count  281.00
43 2021-03-10 22:02:00+08:00  apache02   log.error_count    0.00
44 2021-03-10 22:02:00+08:00  apache02     log.row_count  277.00,                   timestamp        cmdb_id  anomaly_count                                           top_kpis
0 2021-03-10 22:20:00+08:00           IG01             19  OSLinux-CPU_CPU_CPUidleutil|max=74.5406|p95=74...
1 2021-03-10 22:20:00+08:00           IG02             33  JVM-Memory_7778_JVM_Memory_HeapMemoryMax|max=2...
2 2021-03-10 22:20:00+08:00           MG01             35  JVM-Memory_7779_JVM_Memory_HeapMemoryUsed|max=...
3 2021-03-10 22:20:00+08:00           MG02             44  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRTp...
4 2021-03-10 22:20:00+08:00        Mysql01            100  Mysql-MySQL_3306_Binlog stmt cache use|max=360...
5 2021-03-10 22:20:00+08:00        Mysql02             95  OSLinux-CPU_CPU_CPULoad|max=0.91|p95=0.905|rat...
6 2021-03-10 22:20:00+08:00        Redis01             43  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSCapac...
7 2021-03-10 22:20:00+08:00        Redis02             40  OSLinux-NTP_197.30.1.68_NtpServerTimeOffset|ma...
8 2021-03-10 22:20:00+08:00   ServiceTest1              2  app.rr|max=100|p95=100|ratio=1.000; app.sr|max...
9 2021-03-10 22:20:00+08:00  ServiceTest10              2  app.rr|max=100|p95=100|ratio=1.000; app.sr|max...,                   timestamp source_cmdb counterpart_cmdb  trace_anom_count  max_over_p95_ratio
0 2021-03-10 22:20:00+08:00    Tomcat03             root                 3             1.01986
1 2021-03-10 22:20:00+08:00        IG01             root                 3             1.00000
2 2021-03-10 22:20:00+08:00        IG02             root                 3             1.00000
3 2021-03-10 22:20:00+08:00        MG01             root                 3             1.00000
4 2021-03-10 22:20:00+08:00        MG02             root                 3             1.00000
5 2021-03-10 22:20:00+08:00    Tomcat01             root                 3             1.00000
6 2021-03-10 22:20:00+08:00    Tomcat02             root                 3             1.00000
7 2021-03-10 22:20:00+08:00    Tomcat04             root                 3             1.00000
8 2021-03-10 22:20:00+08:00    dockerA1             root                 3             1.00000
9 2021-03-10 22:20:00+08:00    dockerA2             root                 3             1.00000,                     timestamp   cmdb_id         log_name  value
287 2021-03-10 22:20:00+08:00  Tomcat01  log.error_count    0.0
288 2021-03-10 22:20:00+08:00  Tomcat01    log.row_count   95.0
289 2021-03-10 22:20:00+08:00  Tomcat02  log.error_count    0.0
290 2021-03-10 22:20:00+08:00  Tomcat02    log.row_count  122.0
291 2021-03-10 22:20:00+08:00  Tomcat03  log.error_count    0.0
292 2021-03-10 22:20:00+08:00  Tomcat03    log.row_count  110.0
293 2021-03-10 22:20:00+08:00  Tomcat04  log.error_count    0.0
294 2021-03-10 22:20:00+08:00  Tomcat04    log.row_count  101.0
295 2021-03-10 22:20:00+08:00  apache01  log.error_count    0.0
296 2021-03-10 22:20:00+08:00  apache01    log.row_count  217.0
297 2021-03-10 22:20:00+08:00  apache02  log.error_count    0.0
298 2021-03-10 22:20:00+08:00  apache02    log.row_count  210.0)```
```

In [ ]:
# In[9]:

import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse df_metric and agg_metric from kernel state
# Ensure timestamp_dt exists and is in Asia/Shanghai
df_metric['timestamp_dt'] = pd.to_datetime(df_metric['timestamp_dt']).dt.tz_convert(tz)

# Target timestamp and components
ts_target = pd.Timestamp('2021-03-10 22:02:00', tz=tz)
components = ['Mysql02', 'Mysql01', 'Redis02', 'Redis01', 'Tomcat03']

# Filter metric rows for the timestamp and components
detailed_metrics_2202 = df_metric[
    (df_metric['timestamp_dt'] == ts_target) &
    (df_metric['cmdb_id'].isin(components))
].copy()

# Join with global p95 from agg_metric (precomputed)
detailed_metrics_2202 = detailed_metrics_2202.merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Compute ratio = value / p95 with handling for p95 == 0 and NaN
detailed_metrics_2202['ratio'] = np.nan
valid_mask = detailed_metrics_2202['p95'].notna() & (detailed_metrics_2202['p95'] != 0)
detailed_metrics_2202.loc[valid_mask, 'ratio'] = detailed_metrics_2202.loc[valid_mask, 'value'] / detailed_metrics_2202.loc[valid_mask, 'p95']
# p95 == 0 and value > 0 => inf
zero_p95_mask = (detailed_metrics_2202['p95'] == 0) & (detailed_metrics_2202['value'] > 0)
detailed_metrics_2202.loc[zero_p95_mask, 'ratio'] = np.inf

# Select and order requested columns, rename timestamp_dt -> timestamp
detailed_metrics_2202 = detailed_metrics_2202[['timestamp_dt', 'cmdb_id', 'kpi_name', 'value', 'p95', 'ratio']].rename(columns={'timestamp_dt':'timestamp'})

# Sort by cmdb_id then descending ratio
detailed_metrics_2202 = detailed_metrics_2202.sort_values(['cmdb_id', 'ratio'], ascending=[True, False]).reset_index(drop=True)

# Keep full result in kernel variable detailed_metrics_2202 and display up to 200 rows as the compact output
detailed_metrics_2202.head(200)

```
Out[9]:
```
Summary for 2021-03-10 22:02:00+08:00 (detailed_metrics_2202 stored in kernel):

- What was produced:
  - All metric rows at 22:02 for components Mysql02, Mysql01, Redis02, Redis01, Tomcat03 were extracted and joined with the global p95 (agg_metric).
  - The full result is saved in kernel variable detailed_metrics_2202 and contains up to 200 rows (200 rows returned).

- Key findings (top offenders by ratio):
  - Highest ratio observed: Mysql01 · Mysql-MySQL_3306_Bytes Received: value = -3056.0333, p95 = -2299.4922, ratio ≈ 1.329 — note this metric is negative (possible metric sign/interpretation issue).
  - Other notable Mysql01 KPIs with high ratios: local disk read/write metrics (ratios ~1.10–1.20) and binlog/cache metrics at ratio = 1.0.
  - Mysql02 shows many filesystem and MySQL I/O related KPIs with ratio = 1.0 (many KPIs exactly at their global p95).
  - Redis01/Redis02 and Tomcat03 also have multiple KPIs at or above p95 (many ratios = 1.0), indicating widespread, persistent breaches across many KPIs at this minute.

- Interpretation & recommended next steps:
  1. Investigate MySQL hosts first (Mysql01, Mysql02): high counts and several I/O / binlog / filesystem KPIs are at-or-above p95 — look into InnoDB I/O, binlog activity, and disk performance.
  2. Check the negative-valued metric (Bytes Received) for Mysql01 — confirm metric semantics and p95 calculation (negative values may indicate counter wrap, delta sign convention, or collection issue).
  3. For Redis and Tomcat, many KPIs equal p95 (ratio ≈ 1.0) across the minute — evaluate whether p95 thresholds are appropriate or these represent a true sustained condition.
  4. Correlate these metric rows with the trace anomalies and logs already collected for 22:02 to identify whether the metric breaches correspond with request spikes, GC activity, or disk saturation.

If you want, I can:
- show the top 20 rows from detailed_metrics_2202,
- compute per-component KPI counts from that table,
- or extract the rows for any single component (e.g., Mysql01) for deeper inspection. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

timestamp  cmdb_id                                           kpi_name         value           p95     ratio
0   2021-03-10 22:02:00+08:00  Mysql01                    Mysql-MySQL_3306_Bytes Received  -3056.033300  -2299.492225  1.329004
1   2021-03-10 22:02:00+08:00  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...      6.000000      5.000000  1.200000
2   2021-03-10 22:02:00+08:00  Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...     17.000000     15.500000  1.096774
3   2021-03-10 22:02:00+08:00  Mysql01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKTps     16.000000     15.550000  1.028939
4   2021-03-10 22:02:00+08:00  Mysql01             Mysql-MySQL_3306_Binlog stmt cache use    360.000000    360.000000  1.000000
..                        ...      ...                                                ...           ...           ...       ...
195 2021-03-10 22:02:00+08:00  Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysql_FSAvailableS...  11648.000000  11648.000000  1.000000
196 2021-03-10 22:02:00+08:00  Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysql_FSInodeUsedP...      1.877238      1.877238  1.000000
197 2021-03-10 22:02:00+08:00  Mysql02      OSLinux-OSLinux_FILESYSTEM_-mysql_FSUsedSpace   2622.000000   2622.000000  1.000000
198 2021-03-10 22:02:00+08:00  Mysql02    OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSCapacity    100.000000    100.000000  1.000000
199 2021-03-10 22:02:00+08:00  Mysql02  OSLinux-OSLinux_FILESYSTEM_-mysqlbak_FSInodeUs...      0.005646      0.005646  1.000000

[200 rows x 6 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import re
import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables where possible: df_trace, agg_trace, df_metric, agg_metric
tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware in Asia/Shanghai
for df in (df_trace, df_metric):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt']).dt.tz_convert(tz)

# Target timestamps
ts_list = [pd.Timestamp('2021-03-10 22:02:00', tz=tz), pd.Timestamp('2021-03-10 22:20:00', tz=tz)]

# 1) Filter trace rows at target timestamps and join global p95
trace_sel = df_trace[df_trace['timestamp_dt'].isin(ts_list)].merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
).copy()

# Keep only rows where p95 exists and value >= p95 (trace anomalies)
trace_sel = trace_sel[trace_sel['p95'].notna()].copy()
trace_sel['is_anom'] = trace_sel['value'] >= trace_sel['p95']

trace_anoms = trace_sel[trace_sel['is_anom']].copy()

# 2) Parse counterpart from trace_name (to_ or from_) or set 'UNKNOWN'
def extract_counterpart(name):
    if pd.isna(name):
        return 'UNKNOWN'
    m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)', name)
    return m.group(1) if m else 'UNKNOWN'

trace_anoms['counterpart_cmdb'] = trace_anoms['trace_name'].apply(extract_counterpart)

# Compute ratio with p95 handling
trace_anoms['ratio'] = np.nan
valid_mask = trace_anoms['p95'].notna() & (trace_anoms['p95'] != 0)
trace_anoms.loc[valid_mask, 'ratio'] = trace_anoms.loc[valid_mask, 'value'] / trace_anoms.loc[valid_mask, 'p95']
zero_mask = (trace_anoms['p95'] == 0) & (trace_anoms['value'] > 0)
trace_anoms.loc[zero_mask, 'ratio'] = np.inf

# 3) Aggregate trace anomalies by (timestamp, counterpart_cmdb)
trace_grp = trace_anoms.groupby(['timestamp_dt', 'counterpart_cmdb'], as_index=False).agg(
    trace_anom_count=('trace_name', 'size'),
    example_trace_names=('trace_name', lambda s: ";".join(sorted(pd.unique(s))[:3])),
    max_trace_ratio=('ratio', 'max')
).rename(columns={'timestamp_dt':'timestamp'})

# 4) For each (timestamp, counterpart) compute metric anomalies for counterpart at same timestamp
# Prepare metric rows at target timestamps joined with agg_metric
metric_sel = df_metric[df_metric['timestamp_dt'].isin(ts_list)].merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
).copy()

metric_sel = metric_sel[metric_sel['p95'].notna()].copy()
metric_sel['is_anom'] = metric_sel['value'] >= metric_sel['p95']

# Compute ratio for metric rows
metric_sel['ratio'] = np.nan
valid_mask_m = metric_sel['p95'].notna() & (metric_sel['p95'] != 0)
metric_sel.loc[valid_mask_m, 'ratio'] = metric_sel.loc[valid_mask_m, 'value'] / metric_sel.loc[valid_mask_m, 'p95']
zero_mask_m = (metric_sel['p95'] == 0) & (metric_sel['value'] > 0)
metric_sel.loc[zero_mask_m, 'ratio'] = np.inf

# Keep only anomalous metric rows
metric_anoms = metric_sel[metric_sel['is_anom']].copy()

# Aggregate metric anomalies by (timestamp, counterpart_cmdb) where counterpart_cmdb is the metric cmdb_id
metric_grp = metric_anoms.groupby(['timestamp_dt', 'cmdb_id'], as_index=False).agg(
    counterpart_metric_anom_count=('kpi_name', 'size'),
    # example_metric_kpis: up to 3 KPI entries formatted as kpi_name|max=value|p95=...|ratio=...
    example_metric_kpis=('kpi_name', lambda names: None)  # placeholder, fill next
).rename(columns={'timestamp_dt':'timestamp', 'cmdb_id':'counterpart_cmdb'})

# Build example_metric_kpis properly by grouping
def mk_example_metric_kpis(sub):
    # take up to 3 distinct kpis sorted by ratio desc
    sub = sub.copy()
    sub = sub.sort_values('ratio', ascending=False)
    rows = []
    for _, r in sub[['kpi_name','value','p95','ratio']].drop_duplicates(subset=['kpi_name']).head(3).iterrows():
        ratio = r['ratio']
        ratio_str = "inf" if np.isinf(ratio) else (f"{ratio:.3f}" if not pd.isna(ratio) else "NaN")
        rows.append(f"{r['kpi_name']}|max={r['value']:.6g}|p95={r['p95']:.6g}|ratio={ratio_str}")
    return ";".join(rows)

# Apply grouping to compute example_metric_kpis
if not metric_anoms.empty:
    metric_examples = metric_anoms.groupby(['timestamp_dt','cmdb_id']).apply(lambda df: mk_example_metric_kpis(df)).reset_index()
    metric_examples.columns = ['timestamp', 'counterpart_cmdb', 'example_metric_kpis']
    metric_counts = metric_anoms.groupby(['timestamp_dt','cmdb_id']).size().reset_index(name='counterpart_metric_anom_count')
    metric_counts.columns = ['timestamp','counterpart_cmdb','counterpart_metric_anom_count']
    metric_grp = metric_counts.merge(metric_examples, on=['timestamp','counterpart_cmdb'], how='left')
else:
    # empty structure
    metric_grp = pd.DataFrame(columns=['timestamp','counterpart_cmdb','counterpart_metric_anom_count','example_metric_kpis'])

# 5) Merge trace_grp and metric_grp on timestamp and counterpart_cmdb
trace_counterpart_summary = pd.merge(
    trace_grp,
    metric_grp,
    on=['timestamp','counterpart_cmdb'],
    how='outer'
)

# Fill NaNs for counts with 0 and keep example fields as empty string if missing
trace_counterpart_summary['trace_anom_count'] = trace_counterpart_summary['trace_anom_count'].fillna(0).astype(int)
trace_counterpart_summary['max_trace_ratio'] = trace_counterpart_summary['max_trace_ratio'].replace({np.nan: np.nan})  # keep NaN if missing
trace_counterpart_summary['counterpart_metric_anom_count'] = trace_counterpart_summary['counterpart_metric_anom_count'].fillna(0).astype(int)
trace_counterpart_summary['example_trace_names'] = trace_counterpart_summary.get('example_trace_names', pd.Series(dtype=object)).fillna('')
trace_counterpart_summary['example_metric_kpis'] = trace_counterpart_summary.get('example_metric_kpis', pd.Series(dtype=object)).fillna('')

# If some counterpart entries appear in metric_grp but not in trace_grp, ensure example_trace_names exists
if 'example_trace_names' not in trace_counterpart_summary.columns:
    trace_counterpart_summary['example_trace_names'] = ''

# Compute combined count for sorting
trace_counterpart_summary['combined_count'] = trace_counterpart_summary['trace_anom_count'] + trace_counterpart_summary['counterpart_metric_anom_count']

# Sort by timestamp asc then combined_count desc
trace_counterpart_summary = trace_counterpart_summary.sort_values(['timestamp', 'combined_count'], ascending=[True, False]).reset_index(drop=True)

# Limit to top 20 rows for output and store full table in kernel variable
trace_counterpart_summary = trace_counterpart_summary[[
    'timestamp','counterpart_cmdb','trace_anom_count','max_trace_ratio',
    'counterpart_metric_anom_count','example_trace_names','example_metric_kpis'
]]

# Store in kernel variable and display compact top 20 rows
trace_counterpart_summary.head(20)

```
Out[10]:
```
Summary of trace→counterpart findings for 2021-03-10 22:02:00+08:00 (compact):

- Highest metric-only impact:
  - Mysql02 — 116 metric KPI breaches at 22:02 (many MySQL I/O / filesystem KPIs). trace_anom_count = 0.
  - Mysql01 — 103 metric KPI breaches (MySQL I/O/binlog). trace_anom_count = 0.
  - Redis02 — 65 metric KPI breaches. trace_anom_count = 0.
  - Redis01 — 55 metric KPI breaches. trace_anom_count = 0.
  - Tomcat03 — 50 metric KPI breaches. trace_anom_count = 0.
  => Interpretation: DB and cache nodes show the largest metric-volume anomalies at this minute; prioritize MySQL/Redis investigation (I/O, binlog activity, disk).

- Counterparts with both trace and metric signals (higher priority for correlation):
  - MG02 — trace_anom_count = 4, max_trace_ratio ≈ 1.09; counterpart_metric_anom_count = 12. Example traces: from/to MG02; example metric KPIs: network/connection and disk I/O. Suggest correlated trace+metric investigation on MG02.
  - MG01 — trace_anom_count = 3, max_trace_ratio ≈ 1.015; counterpart_metric_anom_count = 14. Example traces include from/to MG01; metrics show CPU and related KPIs.
  - IG01 — trace_anom_count = 2, max_trace_ratio ≈ 1.056; counterpart_metric_anom_count = 31. Example trace: trace.from_IG01.row_count; metrics include disk and memory KPIs.
  - Tomcat02 & Tomcat04 — each trace_anom_count = 2, max_trace_ratio ≈ 1.036–1.056; counterpart_metric_anom_count = 26. Trace names reference Tomcat↔other components; metrics show memory/disk related breaches.
  => Interpretation: MG01/MG02, IG01, Tomcat02/04 show correlated trace + metric anomalies and should be investigated as likely propagation or downstream-impact points.

- Counterparts with trace anomalies but no metric anomalies:
  - dockerA1 — trace_anom_count = 3, max_trace_ratio ≈ 1.091, counterpart_metric_anom_count = 0.
  - Some dockerB* entries appear in traces but have few/no metric breaches at this minute.
  => Interpretation: tracing indicates potential latency / request-path issues involving dockerA1 without local resource saturation — investigate networking, routing, or service endpoints on dockerA1.

- Notes from examples:
  - example_trace_names and example_metric_kpis columns contain the representative trace/metric names (up to 3) for quick inspection (e.g., InnoDB write metrics for MySQL, LocalDisk DSKRead/Tps for disk activity).
  - Many metric breaches are exactly at p95 (ratio ≈ 1.0), indicating sustained high baseline load; some trace ratios are modest (~1.03–1.09), not huge spikes but consistent.

Recommended next steps (concise):
1. Prioritize MySQL (Mysql02, Mysql01) and Redis (Redis02, Redis01) for disk/I/O and DB-level checks (InnoDB writes, binlog, filesystem).
2. Investigate MG02 and MG01 (correlated trace+metric anomalies) and IG01 and Tomcat02/04 for cross-component propagation or request handling issues.
3. Inspect dockerA1 traces (trace anomalies with no metric faults) for network/latency or routing issues.
4. Use the example_trace_names and example_metric_kpis from the table to drill into the specific trace spans and KPI time series / logs for root-cause.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp counterpart_cmdb  trace_anom_count  max_trace_ratio  counterpart_metric_anom_count                                example_trace_names                                example_metric_kpis
0  2021-03-10 22:02:00+08:00          Mysql02                 0              NaN                            116                                                     Mysql-MySQL_3306_Innodb data written|max=29588...
1  2021-03-10 22:02:00+08:00          Mysql01                 0              NaN                            103                                                     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
2  2021-03-10 22:02:00+08:00          Redis02                 0              NaN                             65                                                     OSLinux-CPU_CPU_CPUSysTime|max=0.645699|p95=0....
3  2021-03-10 22:02:00+08:00          Redis01                 0              NaN                             55                                                     OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace|...
4  2021-03-10 22:02:00+08:00         Tomcat03                 0              NaN                             50                                                     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTp...
5  2021-03-10 22:02:00+08:00             IG02                 0              NaN                             34                                                     OSLinux-OSLinux_MEMORY_MEMORY_CacheMem|max=543...
6  2021-03-10 22:02:00+08:00             IG01                 2         1.055901                             31                          trace.from_IG01.row_count  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...
7  2021-03-10 22:02:00+08:00         Tomcat02                 2         1.035857                             26  trace.from_Tomcat02.row_count;trace.to_Tomcat0...  OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem|max=1...
8  2021-03-10 22:02:00+08:00         Tomcat04                 2         1.055901                             26  trace.from_Tomcat04.row_count;trace.to_Tomcat0...  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem|max=420...
9  2021-03-10 22:02:00+08:00         apache01                 0              NaN                             25                                                     OSLinux-CPU_CPU-3_SingleCpuUtil|max=2.3082|p95...
10 2021-03-10 22:02:00+08:00         Tomcat01                 0              NaN                             19                                                     Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory|max=2....
11 2021-03-10 22:02:00+08:00             MG01                 3         1.014975                             14  trace.from_MG01.row_count;trace.to_MG01.row_count  OSLinux-CPU_CPU_CPUCpuUtil|max=26.9843|p95=26....
12 2021-03-10 22:02:00+08:00             MG02                 4         1.090652                             12  trace.from_MG02.row_count;trace.to_MG02.durati...  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...
13 2021-03-10 22:02:00+08:00         apache02                 0              NaN                             16                                                     OSLinux-CPU_CPU_CPUSysTime|max=0.461021|p95=0....
14 2021-03-10 22:02:00+08:00    ServiceTest10                 0              NaN                              3                                                     app.mrt|max=0.10646|p95=0.095875|ratio=1.110;a...
15 2021-03-10 22:02:00+08:00    ServiceTest11                 0              NaN                              3                                                     app.cnt|max=101|p95=77.5|ratio=1.303;app.rr|ma...
16 2021-03-10 22:02:00+08:00     ServiceTest2                 0              NaN                              3                                                     app.cnt|max=88|p95=71|ratio=1.239;app.rr|max=1...
17 2021-03-10 22:02:00+08:00         dockerA1                 3         1.090652                              0  trace.from_dockerA1.duration_p95;trace.to_dock...                                                   
18 2021-03-10 22:02:00+08:00         dockerB1                 0              NaN                              3                                                     Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...
19 2021-03-10 22:02:00+08:00     ServiceTest1                 0              NaN                              2                                                     app.rr|max=100|p95=100|ratio=1.000;app.sr|max=...```
```

In [ ]:
# In[11]:

import re
import numpy as np
import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Ensure timestamp_dt exists and is timezone-aware
for df in (df_trace, df_metric):
    if 'timestamp_dt' not in df.columns:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
    else:
        df['timestamp_dt'] = pd.to_datetime(df['timestamp_dt']).dt.tz_convert(tz)

if 'fault_start_ts' in metric_faults.columns:
    metric_faults['fault_start_ts'] = pd.to_datetime(metric_faults['fault_start_ts']).dt.tz_convert(tz)

# Target timestamps
ts_list = [pd.Timestamp('2021-03-10 22:02:00', tz=tz), pd.Timestamp('2021-03-10 22:20:00', tz=tz)]

# Helper to extract counterpart
def extract_counterpart(name):
    if pd.isna(name):
        return 'UNKNOWN'
    m = re.search(r'\b(?:to|from)_([A-Za-z0-9]+)', name)
    return m.group(1) if m else 'UNKNOWN'

# 1) Filter trace rows at target timestamps and join with agg_trace
trace_sel = df_trace[df_trace['timestamp_dt'].isin(ts_list)].merge(
    agg_trace[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
).copy()

# Keep only trace anomaly rows where p95 exists and value >= p95
trace_sel = trace_sel[trace_sel['p95'].notna()].copy()
trace_sel['is_anom'] = trace_sel['value'] >= trace_sel['p95']
trace_anoms = trace_sel[trace_sel['is_anom']].copy()

# parse counterpart and compute ratio
trace_anoms['counterpart_cmdb'] = trace_anoms['trace_name'].apply(extract_counterpart)
trace_anoms['ratio'] = np.nan
valid_mask = trace_anoms['p95'].notna() & (trace_anoms['p95'] != 0)
trace_anoms.loc[valid_mask, 'ratio'] = trace_anoms.loc[valid_mask, 'value'] / trace_anoms.loc[valid_mask, 'p95']
zero_mask = (trace_anoms['p95'] == 0) & (trace_anoms['value'] > 0)
trace_anoms.loc[zero_mask, 'ratio'] = np.inf

# 2) Group trace anomalies by (timestamp, counterpart_cmdb)
trace_grp = trace_anoms.groupby(['timestamp_dt', 'counterpart_cmdb'], as_index=False).agg(
    trace_anom_count=('trace_name', 'size'),
    example_trace_names=('trace_name', lambda s: ";".join(sorted(pd.unique(s))[:3])),
    max_trace_ratio=('ratio', 'max')
).rename(columns={'timestamp_dt':'timestamp'})

# 3) Prepare metric anomalies at the same timestamps
metric_sel = df_metric[df_metric['timestamp_dt'].isin(ts_list)].merge(
    agg_metric[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
).copy()

metric_sel = metric_sel[metric_sel['p95'].notna()].copy()
metric_sel['is_anom'] = metric_sel['value'] >= metric_sel['p95']

# compute ratio for metrics
metric_sel['ratio'] = np.nan
valid_mask_m = metric_sel['p95'].notna() & (metric_sel['p95'] != 0)
metric_sel.loc[valid_mask_m, 'ratio'] = metric_sel.loc[valid_mask_m, 'value'] / metric_sel.loc[valid_mask_m, 'p95']
zero_mask_m = (metric_sel['p95'] == 0) & (metric_sel['value'] > 0)
metric_sel.loc[zero_mask_m, 'ratio'] = np.inf

metric_anoms = metric_sel[metric_sel['is_anom']].copy()

# Aggregate metric anomalies per (timestamp, counterpart_cmdb)
if not metric_anoms.empty:
    metric_counts = metric_anoms.groupby(['timestamp_dt','cm_id' if 'cm_id' in metric_anoms.columns else 'cmdb_id'], as_index=False).size()
    # The above grouping might use 'cmdb_id' normally; ensure correct column name
    if 'cm_id' in metric_anoms.columns:
        # rename to expected names
        metric_counts = metric_anoms.groupby(['timestamp_dt','cm_id'], as_index=False).size()
        metric_counts.columns = ['timestamp_dt','counterpart_cmdb','counterpart_metric_anom_count']
    else:
        metric_counts = metric_anoms.groupby(['timestamp_dt','cmdb_id'], as_index=False).size()
        metric_counts.columns = ['timestamp_dt','counterpart_cmdb','counterpart_metric_anom_count']
    metric_counts = metric_counts.rename(columns={'timestamp_dt':'timestamp'})
    # Example metric KPI (highest ratio) per (timestamp, counterpart)
    def mk_example_metric(df):
        row = df.sort_values('ratio', ascending=False).iloc[0]
        ratio = row['ratio']
        ratio_str = "inf" if np.isinf(ratio) else (f"{ratio:.3f}" if not pd.isna(ratio) else "NaN")
        return f"{row['kpi_name']}|value={row['value']:.6g}|p95={row['p95']:.6g}|ratio={ratio_str}"
    metric_examples = metric_anoms.groupby(['timestamp_dt', 'cmdb_id']).apply(lambda d: mk_example_metric(d)).reset_index()
    metric_examples.columns = ['timestamp','counterpart_cmdb','example_metric_kpi']
    # Merge counts and examples
    metric_grp = pd.merge(metric_counts, metric_examples, on=['timestamp','counterpart_cmdb'], how='left')
else:
    metric_grp = pd.DataFrame(columns=['timestamp','counterpart_cmdb','counterpart_metric_anom_count','example_metric_kpi'])

# 3b) earliest_metric_fault_start from metric_faults per counterpart
if not metric_faults.empty:
    earliest_fault_by_comp = metric_faults.groupby('cmdb_id', as_index=False).agg(earliest_metric_fault_start=('fault_start_ts','min'))
else:
    earliest_fault_by_comp = pd.DataFrame(columns=['cmdb_id','earliest_metric_fault_start'])

# 4) Determine downstream_only flag: counterpart appears in trace_anoms as counterpart but NOT as a source cmdb_id in trace_anoms at same timestamp
sources_per_ts = trace_anoms.groupby('timestamp_dt')['cmdb_id'].apply(lambda arr: set(arr)).to_dict()

# 5) Merge trace_grp and metric_grp and attach earliest fault and downstream_only
combined = pd.merge(trace_grp, metric_grp, on=['timestamp','counterpart_cmdb'], how='outer')

# fill missing counts with 0 and example fields with empty strings
combined['trace_anom_count'] = combined['trace_anom_count'].fillna(0).astype(int)
combined['max_trace_ratio'] = combined['max_trace_ratio'].where(pd.notna(combined['max_trace_ratio']), np.nan)
combined['counterpart_metric_anom_count'] = combined.get('counterpart_metric_anom_count', pd.Series(dtype=int)).fillna(0).astype(int)
combined['example_trace_names'] = combined.get('example_trace_names', pd.Series(dtype=object)).fillna('')
combined['example_metric_kpi'] = combined.get('example_metric_kpi', pd.Series(dtype=object)).fillna('')

# attach earliest_metric_fault_start
combined = combined.merge(earliest_fault_by_comp.rename(columns={'cmdb_id':'counterpart_cmdb'}), on='counterpart_cmdb', how='left')

# compute downstream_only
def is_downstream_only(row):
    ts = row['timestamp']
    srcs = sources_per_ts.get(ts, set())
    return row['counterpart_cmdb'] not in srcs

combined['downstream_only'] = combined.apply(is_downstream_only, axis=1)

# select and order columns
final_candidates_for_minutes = combined[[
    'timestamp','counterpart_cmdb','downstream_only','trace_anom_count','max_trace_ratio',
    'counterpart_metric_anom_count','earliest_metric_fault_start','example_metric_kpi'
]].copy()

# compute combined score for sorting: trace_anom_count + counterpart_metric_anom_count
final_candidates_for_minutes['combined_count'] = final_candidates_for_minutes['trace_anom_count'] + final_candidates_for_minutes['counterpart_metric_anom_count']

# sort by timestamp asc then combined_count desc
final_candidates_for_minutes = final_candidates_for_minutes.sort_values(['timestamp','combined_count'], ascending=[True, False]).reset_index(drop=True)

# limit to top 20 rows for output
final_candidates_for_minutes = final_candidates_for_minutes.head(20)

# Store in kernel variable (already assigned) and display compact result
final_candidates_for_minutes

```
Out[11]:
```
Summary of ranked per-minute counterpart candidates (22:02 and 22:20 UTC+8) — key findings and actionable hints:

Top candidates at 2021-03-10 22:02:00+08:00 (highest combined evidence first)
- Mysql02 — combined_count 116 (trace_anom_count 0, metric_anom_count 116). downstream_only = True. earliest_metric_fault_start = 2021-03-10 22:00. Example KPI: Mysql-MySQL_3306_Innodb data written → suggests heavy InnoDB / disk I/O activity.
- Mysql01 — combined_count 103 (0 trace, 103 metric). downstream_only = True. earliest 22:00. Example KPI: local disk DSKRead/Tps → suggests disk I/O / DB activity.
- Redis02 — combined_count 65 (0 trace, 65 metric). downstream_only = True. earliest 22:00. Example KPI: CPU SysTime / filesystem metrics → resource/filesystem pressure.
- Redis01 — combined_count 55 (0 trace, 55 metric). downstream_only = True. earliest 22:00. Example KPI: filesystem available space metric shown.
- Tomcat03 — combined_count 50 (0 trace, 50 metric). downstream_only = True. earliest 22:00. Example KPI: local disk write TPS → disk-related symptom.
- IG01, IG02, MG01, MG02, Tomcat02, Tomcat04 also appear with mixed trace + metric evidence:
  - MG02: trace_anom_count 4, max_trace_ratio ≈1.09, metric_anom_count 12, earliest 22:00 → trace+metric correlation (network/conn & disk signs).
  - MG01: trace_anom_count 3, metric_anom_count 14 → CPU signal plus trace links.
  - IG01: trace_anom_count 2, metric_anom_count 31 → disk/memory metrics with trace links.

Notable trace-only or downstream-only patterns at 22:02
- dockerA1 — trace_anom_count 3, metric_anom_count 0, not downstream_only (it also appears as a trace source) — indicates trace-side latency or routing issues involving dockerA1, but no local metric saturation at that minute.
- Many high-volume metric anomalies (MySQL, Redis) are downstream_only = True (they are counterparts referenced by traces but not acting as trace sources), making them strong root-cause candidates.

Top candidates at 2021-03-10 22:20:00+08:00 (summary from earlier results)
- Similar pattern: Mysql01/Mysql02 and MG*/IG* and Redis* show prominent metric breaches; many traces reference a counterpart labeled "root" (needs inspection).

Interpretation and likely root cause
- The strongest signals point to MySQL hosts (Mysql02, Mysql01) and Redis hosts as primary suspects — very high counts of metric KPIs breaching global p95, many starting at ~22:00. The example KPIs point to heavy InnoDB activity, binlog/IO, local disk read/write TPS and filesystem metrics — i.e., database I/O or storage pressure.
- MG02 / MG01 and IG01 show correlated trace and metric anomalies (trace links + metric breaches), making them secondary suspects for propagation or intermediate impact.
- dockerA1 has trace anomalies without local metric faults — likely a downstream service endpoint or network/latency issue rather than local resource exhaustion.

Recommended next actions (concise)
1. Prioritize investigation of Mysql02 and Mysql01 (InnoDB, binlog, disk I/O, storage metrics), then Redis01/02.
2. Correlate MySQL/Redis I/O spikes with system-level disk metrics (iostat), host CPU, and Tomcat request spikes.
3. Inspect MG02/MG01 and IG01 for propagated effects (traces mapping to disk/network KPIs).
4. Investigate dockerA1 traces (trace-only anomalies) for network routing or service latency issues.
5. Use the example_metric_kpi strings in the table to drill into exact KPI time series, host logs, and trace spans for root-cause confirmation.

If you want, I can produce a single ranked root-cause recommendation statement (component, earliest occurrence, probable reason) based on these findings. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

timestamp counterpart_cmdb  downstream_only  trace_anom_count  max_trace_ratio  counterpart_metric_anom_count earliest_metric_fault_start                                 example_metric_kpi  combined_count
0  2021-03-10 22:02:00+08:00          Mysql02             True                 0              NaN                            116   2021-03-10 22:00:00+08:00  Mysql-MySQL_3306_Innodb data written|value=295...             116
1  2021-03-10 22:02:00+08:00          Mysql01             True                 0              NaN                            103   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...             103
2  2021-03-10 22:02:00+08:00          Redis02             True                 0              NaN                             65   2021-03-10 22:00:00+08:00  OSLinux-CPU_CPU_CPUSysTime|value=0.645699|p95=...              65
3  2021-03-10 22:02:00+08:00          Redis01             True                 0              NaN                             55   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace|...              55
4  2021-03-10 22:02:00+08:00         Tomcat03             True                 0              NaN                             50   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWTp...              50
5  2021-03-10 22:02:00+08:00             IG02             True                 0              NaN                             34   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem|value=5...              34
6  2021-03-10 22:02:00+08:00             IG01            False                 2         1.055901                             31   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...              33
7  2021-03-10 22:02:00+08:00         Tomcat02            False                 2         1.035857                             26   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_MEMORY_MEMORY_MEMFreeMem|value...              28
8  2021-03-10 22:02:00+08:00         Tomcat04            False                 2         1.055901                             26   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_MEMORY_MEMORY_CacheMem|value=4...              28
9  2021-03-10 22:02:00+08:00         apache01             True                 0              NaN                             25   2021-03-10 22:00:00+08:00  OSLinux-CPU_CPU-3_SingleCpuUtil|value=2.3082|p...              25
10 2021-03-10 22:02:00+08:00         Tomcat01             True                 0              NaN                             19   2021-03-10 22:00:00+08:00  Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory|value=...              19
11 2021-03-10 22:02:00+08:00             MG01            False                 3         1.014975                             14   2021-03-10 22:00:00+08:00  OSLinux-CPU_CPU_CPUCpuUtil|value=26.9843|p95=2...              17
12 2021-03-10 22:02:00+08:00             MG02            False                 4         1.090652                             12   2021-03-10 22:00:00+08:00  OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNu...              16
13 2021-03-10 22:02:00+08:00         apache02             True                 0              NaN                             16   2021-03-10 22:00:00+08:00  OSLinux-CPU_CPU_CPUSysTime|value=0.461021|p95=...              16
14 2021-03-10 22:02:00+08:00    ServiceTest10             True                 0              NaN                              3   2021-03-10 22:00:00+08:00     app.mrt|value=0.10646|p95=0.095875|ratio=1.110               3
15 2021-03-10 22:02:00+08:00    ServiceTest11             True                 0              NaN                              3   2021-03-10 22:00:00+08:00             app.cnt|value=101|p95=77.5|ratio=1.303               3
16 2021-03-10 22:02:00+08:00     ServiceTest2             True                 0              NaN                              3   2021-03-10 22:00:00+08:00                app.cnt|value=88|p95=71|ratio=1.239               3
17 2021-03-10 22:02:00+08:00         dockerA1            False                 3         1.090652                              0                         NaT                                                                  3
18 2021-03-10 22:02:00+08:00         dockerB1             True                 0              NaN                              3                         NaT  Container-DOCKER_CONTAINER_9e8c309d0aab--bcou-...               3
19 2021-03-10 22:02:00+08:00     ServiceTest1             True                 0              NaN                              2   2021-03-10 22:00:00+08:00               app.rr|value=100|p95=100|ratio=1.000               2```
```